# 🌍 Global Share Market Lab

*Developed by **Anish Luitel** · The Half Layer*

**Layout · statistics · advanced formulas · dashboard · explanations — in one notebook.**

This notebook downloads (or loads) prices for a **universe of indices and stocks** (presets: `global` 15 indices + 20 mega-caps, `canada` TSX-focused, or `custom` – your own tickers), optionally converts everything to one **base currency**, then walks through the analysis a long-term investor needs:

| # | Section | What you get |
|---|---------|--------------|
| 1 | **World overview** | Return heat-map, rebased performance, market breadth / regime |
| 2 | **Trend & momentum** | SMA / EMA, golden & death cross, MACD, RSI, Stochastic, ROC, signal table |
| 3 | **Volatility & risk** | Bollinger, ATR, EWMA vol, Sharpe, Sortino, Max-Drawdown, Calmar, VaR / CVaR (3 methods) |
| 4 | **Cross-market** | Correlation (clustered), PCA "how synchronised is the world", Beta / Alpha / Information Ratio, relative strength |
| 5 | **Portfolio** | Efficient frontier, min-variance & max-Sharpe portfolios with weight caps |
| 6 | **Valuation & fundamentals** | P/E, P/B, yields, ROE, DCF with sensitivity grid, 4-factor **scorecard** |
| 7 | **Backtest** | Does a trend rule actually beat buy-and-hold? Costs, drawdowns, basket test |
| 8 | **Dashboard** | One standalone, interactive `HTML` file you can share |

### How to use
1. **Run all cells** (*Kernel → Restart & Run All*). The default uses free Yahoo Finance data through `yfinance`.
2. To change anything — tickers, benchmark, period, risk-free rate, data source — edit the **CONFIG** and **UNIVERSE** cells and re-run.
3. The dashboard is written to `outputs/global_market_dashboard.html`.

> ⚠️ **Educational analysis tool, not financial advice.** Free data can be delayed, adjusted or occasionally wrong. Past performance does not predict future results.

In [ ]:
# ---- Imports & global plotting style -------------------------------------------------
import os, json, math, hashlib, warnings, datetime as dt
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline
from plotly.subplots import make_subplots
from scipy import stats, optimize
from scipy.cluster import hierarchy as sch
from scipy.spatial.distance import squareform
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)

plt.rcParams.update({
    "figure.figsize": (11, 4.5), "figure.dpi": 100, "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
})
PLOTLY_TEMPLATE = "plotly_white"
pio.templates.default = PLOTLY_TEMPLATE
REGION_COLORS = {"North America": "#2563eb", "Europe": "#059669", "Asia-Pacific": "#dc2626",
                 "Latin America": "#d97706", "Canada": "#be123c", "United States": "#1d4ed8", "Other": "#6b7280"}
print("Libraries ready.")

## ⚙️ Configuration — *the only cell you normally need to edit*

| Key | Meaning |
|-----|---------|
| `data_source` | `"yfinance"` free live data · `"csv"` your own files in `csv_folder` · `"sample"` synthetic demo data |
| `period` / `start` / `end` | History window. `period="5y"`, or set `start="2015-01-01"` |
| `universe` | `"global"` (15 indices + 20 mega-caps) · `"canada"` (TSX blue-chips + world context, CAD) · `"mega"` (110 of the largest US, Canadian, European and Asian companies, USD) · `"penny"` (stocks under 5, found live by Yahoo's screener at every rebuild) · `"custom"` (your own lists). The launcher's Rebuild buttons set the environment variable `HL_UNIVERSE` to choose one. |
| `watchlist_csv` | Stocks added from the dashboard's **Add a stock** dialog (`data/watchlist.csv`, columns `ticker,name,ccy`). They are added to every universe. |
| `benchmark` | `"auto"` or any ticker: Index used for Beta, Alpha, Information Ratio and relative strength (`^GSPC` = S&P 500, `^GSPTSE` = TSX, `URTH` = MSCI World ETF) |
| `base_currency` | Currency all returns/risk/correlations are measured in. `"auto"`, `"local"` (no conversion) or a code such as `"USD"`, `"CAD"`, `"EUR"`. Technical indicators always use local prices |
| `risk_free_rate` | Annual risk-free rate used in Sharpe / Sortino / Alpha (e.g. `0.04` = 4 %) |
| `confidence` | Confidence level for VaR / CVaR |
| `cross_market_freq` | `"W-FRI"` weekly returns (recommended for cross-market stats — see note in §4) or `"D"` daily |
| `max_weight` | Cap on any single stock in the optimiser |
| `scorecard_weights` | How much Value / Quality / Momentum / Risk count in the composite score |
| `dcf` | Growth, terminal growth and discount rate for the simple DCF |
| `bt_cost_bps` | Transaction cost assumed in the backtest |
| `my_holdings` / `my_holdings_csv` | **Your own stocks.** Inline list and/or `data/my_portfolio.csv` (export it from the dashboard's **MINE** page). Tickers not already in the universe are downloaded and get the full treatment (chart, technicals, analyst rating) |

In [ ]:
CONFIG = {
    # ---------------- data ----------------
    "data_source": "yfinance",        # "yfinance" | "csv" | "sample"
    "period": "5y",                   # 1y 2y 5y 10y max   (ignored when "start" is set)
    "start": None,                    # e.g. "2015-01-01"
    "end": None,
    "csv_folder": "data/csv",         # one file per ticker: <ticker>.csv  (see section 'Switching the data source')
    "fundamentals_csv": "data/fundamentals.csv",
    "use_cache": True,                # re-use today's download instead of hitting the API again
    "cache_dir": "cache",
    "fallback_to_sample": True,       # if the live download fails, continue with synthetic data (clearly flagged)

    # ---------------- analysis ----------------
    "universe": os.environ.get("HL_UNIVERSE", "global"),   # "global" | "canada" | "mega" | "penny" | "custom"  (the launcher's Rebuild buttons set HL_UNIVERSE)
    "benchmark": "auto",              # "auto" = preset default (global: ^GSPC, canada: ^GSPTSE) or any ticker
    "base_currency": "auto",          # "auto" (global: USD, canada: CAD) | "local" (no conversion) | "USD" | "CAD" | "EUR" ...
    "risk_free_rate": 0.04,
    "confidence": 0.95,
    "cross_market_freq": "W-FRI",     # "W-FRI" or "D"
    "focus_ticker": "NVDA",           # used by the single-asset charts in sections 2 and 3
    "trading_days": 252,

    # ---------------- portfolio ----------------
    "max_weight": 0.25,
    "n_random_portfolios": 4000,
    "mu_shrinkage": 0.5,              # 0 = raw historical mean returns, 1 = every asset gets the average return
    "cov_shrinkage": 0.2,             # 0 = raw covariance, 1 = diagonal only

    # ---------------- valuation ----------------
    "scorecard_weights": {"Value": 0.30, "Quality": 0.30, "Momentum": 0.20, "Risk": 0.20},
    "dcf": {"growth_5y": 0.08, "terminal_growth": 0.025, "discount_rate": 0.09},

    # ---------------- backtest ----------------
    "bt_cost_bps": 10,                # trading cost per position change, in basis points (10 = 0.10 %)

    # ---------------- my portfolio ----------------
    "my_holdings": [],                # inline holdings, e.g. [{"ticker": "AAPL", "shares": 10, "cost": 150.0, "date": "2023-05-01"}]
    "watchlist_csv": "data/watchlist.csv",        # extra stocks added from the dashboard's "Add a stock" dialog (columns: ticker,name,ccy); added to EVERY universe
    "data_url": "https://raw.githubusercontent.com/Proxyanish/Half-Layer--Market/data/latest",   # where the daily GitHub build publishes the newest data (the Refresh button of a file opened without the server downloads it); "" = off
    "my_holdings_csv": "data/my_portfolio.csv",   # columns: ticker,shares,cost,date,note  (the dashboard's MINE page exports this file)

    # ---------------- output ----------------
    "output_dir": "outputs",
    "embed_plotlyjs": True,           # True = dashboard works offline (+4 MB); False = loads plotly.js from a CDN
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("Config loaded  →  source:", CONFIG["data_source"], "| period:", CONFIG["period"], "| benchmark:", CONFIG["benchmark"])

### The universe — add, remove or replace any ticker
Tickers follow Yahoo Finance symbols (`.TO` Toronto, `.L` London, `.DE` Frankfurt, `.PA` Paris, `.AS` Amsterdam, `.SW` Swiss, `.T` Tokyo, `.HK` Hong Kong, `.NS` India NSE, `.AX` Australia, `.KS` Korea). Each entry is `"ticker": ("Display name", "Region")`.

In [ ]:
UNIVERSES = {
    "global": {
        "indices": {
            "^GSPC": ("S&P 500", "North America"), "^IXIC": ("Nasdaq Composite", "North America"),
            "^DJI": ("Dow Jones", "North America"), "^GSPTSE": ("S&P/TSX Composite", "North America"),
            "^FTSE": ("FTSE 100", "Europe"), "^GDAXI": ("DAX 40", "Europe"), "^FCHI": ("CAC 40", "Europe"),
            "^STOXX50E": ("Euro Stoxx 50", "Europe"), "^N225": ("Nikkei 225", "Asia-Pacific"),
            "^HSI": ("Hang Seng", "Asia-Pacific"), "000001.SS": ("Shanghai Composite", "Asia-Pacific"),
            "^NSEI": ("Nifty 50", "Asia-Pacific"), "^AXJO": ("ASX 200", "Asia-Pacific"),
            "^KS11": ("KOSPI", "Asia-Pacific"), "^BVSP": ("Bovespa", "Latin America")},
        "stocks": {
            "AAPL": ("Apple", "North America"), "MSFT": ("Microsoft", "North America"), "NVDA": ("NVIDIA", "North America"),
            "AMZN": ("Amazon", "North America"), "JPM": ("JPMorgan Chase", "North America"),
            "SHOP.TO": ("Shopify", "North America"), "RY.TO": ("Royal Bank of Canada", "North America"),
            "ENB.TO": ("Enbridge", "North America"), "ASML.AS": ("ASML", "Europe"), "SAP.DE": ("SAP", "Europe"),
            "NESN.SW": ("Nestlé", "Europe"), "MC.PA": ("LVMH", "Europe"), "SHEL.L": ("Shell", "Europe"),
            "7203.T": ("Toyota", "Asia-Pacific"), "005930.KS": ("Samsung Electronics", "Asia-Pacific"),
            "0700.HK": ("Tencent", "Asia-Pacific"), "9988.HK": ("Alibaba", "Asia-Pacific"),
            "RELIANCE.NS": ("Reliance Industries", "Asia-Pacific"), "TCS.NS": ("Tata Consultancy", "Asia-Pacific"),
            "BHP.AX": ("BHP Group", "Asia-Pacific")},
        "benchmark": "^GSPC", "base_currency": "USD"},
    "mega": {
        "indices": {
            "^GSPC": ("S&P 500", "North America"), "^IXIC": ("Nasdaq Composite", "North America"),
            "^GSPTSE": ("S&P/TSX Composite", "North America"), "^FTSE": ("FTSE 100", "Europe"),
            "^GDAXI": ("DAX 40", "Europe"), "^N225": ("Nikkei 225", "Asia-Pacific"),
            "^HSI": ("Hang Seng", "Asia-Pacific"), "^AXJO": ("ASX 200", "Asia-Pacific"), "^BVSP": ("Bovespa", "Latin America")},
        "stocks": {
            "AAPL": ("Apple", "North America"),
            "MSFT": ("Microsoft", "North America"),
            "NVDA": ("NVIDIA", "North America"),
            "AMZN": ("Amazon", "North America"),
            "GOOGL": ("Alphabet (Google)", "North America"),
            "META": ("Meta Platforms", "North America"),
            "TSLA": ("Tesla", "North America"),
            "BRK-B": ("Berkshire Hathaway", "North America"),
            "AVGO": ("Broadcom", "North America"),
            "LLY": ("Eli Lilly", "North America"),
            "JPM": ("JPMorgan Chase", "North America"),
            "V": ("Visa", "North America"),
            "XOM": ("Exxon Mobil", "North America"),
            "UNH": ("UnitedHealth", "North America"),
            "MA": ("Mastercard", "North America"),
            "COST": ("Costco", "North America"),
            "HD": ("Home Depot", "North America"),
            "PG": ("Procter & Gamble", "North America"),
            "JNJ": ("Johnson & Johnson", "North America"),
            "WMT": ("Walmart", "North America"),
            "NFLX": ("Netflix", "North America"),
            "BAC": ("Bank of America", "North America"),
            "ORCL": ("Oracle", "North America"),
            "CRM": ("Salesforce", "North America"),
            "ABBV": ("AbbVie", "North America"),
            "CVX": ("Chevron", "North America"),
            "KO": ("Coca-Cola", "North America"),
            "MRK": ("Merck", "North America"),
            "AMD": ("AMD", "North America"),
            "PEP": ("PepsiCo", "North America"),
            "ADBE": ("Adobe", "North America"),
            "TMO": ("Thermo Fisher", "North America"),
            "LIN": ("Linde", "North America"),
            "MCD": ("McDonald's", "North America"),
            "CSCO": ("Cisco", "North America"),
            "ACN": ("Accenture", "North America"),
            "ABT": ("Abbott", "North America"),
            "WFC": ("Wells Fargo", "North America"),
            "DIS": ("Disney", "North America"),
            "GE": ("GE Aerospace", "North America"),
            "IBM": ("IBM", "North America"),
            "QCOM": ("Qualcomm", "North America"),
            "TXN": ("Texas Instruments", "North America"),
            "INTU": ("Intuit", "North America"),
            "CAT": ("Caterpillar", "North America"),
            "VZ": ("Verizon", "North America"),
            "AMGN": ("Amgen", "North America"),
            "PFE": ("Pfizer", "North America"),
            "NOW": ("ServiceNow", "North America"),
            "UBER": ("Uber", "North America"),
            "GS": ("Goldman Sachs", "North America"),
            "MS": ("Morgan Stanley", "North America"),
            "T": ("AT&T", "North America"),
            "BLK": ("BlackRock", "North America"),
            "ISRG": ("Intuitive Surgical", "North America"),
            "SPGI": ("S&P Global", "North America"),
            "NEE": ("NextEra Energy", "North America"),
            "LOW": ("Lowe's", "North America"),
            "UNP": ("Union Pacific", "North America"),
            "HON": ("Honeywell", "North America"),
            "RTX": ("RTX", "North America"),
            "PLTR": ("Palantir", "North America"),
            "DE": ("Deere", "North America"),
            "BA": ("Boeing", "North America"),
            "SBUX": ("Starbucks", "North America"),
            "NKE": ("Nike", "North America"),
            "INTC": ("Intel", "North America"),
            "MU": ("Micron", "North America"),
            "PANW": ("Palo Alto Networks", "North America"),
            "TSM": ("TSMC (ADR)", "North America"),
            "RY.TO": ("Royal Bank of Canada", "North America"),
            "TD.TO": ("TD Bank", "North America"),
            "SHOP.TO": ("Shopify", "North America"),
            "ENB.TO": ("Enbridge", "North America"),
            "CNQ.TO": ("Canadian Natural", "North America"),
            "CP.TO": ("CPKC", "North America"),
            "CNR.TO": ("CN Rail", "North America"),
            "BNS.TO": ("Scotiabank", "North America"),
            "BMO.TO": ("Bank of Montreal", "North America"),
            "CSU.TO": ("Constellation Software", "North America"),
            "ATD.TO": ("Couche-Tard", "North America"),
            "SU.TO": ("Suncor", "North America"),
            "ASML.AS": ("ASML", "Europe"),
            "SAP.DE": ("SAP", "Europe"),
            "NESN.SW": ("Nestlé", "Europe"),
            "MC.PA": ("LVMH", "Europe"),
            "SHEL.L": ("Shell", "Europe"),
            "NOVN.SW": ("Novartis", "Europe"),
            "AZN.L": ("AstraZeneca", "Europe"),
            "HSBA.L": ("HSBC", "Europe"),
            "SIE.DE": ("Siemens", "Europe"),
            "OR.PA": ("L'Oréal", "Europe"),
            "TTE.PA": ("TotalEnergies", "Europe"),
            "ULVR.L": ("Unilever", "Europe"),
            "RMS.PA": ("Hermès", "Europe"),
            "SAN.PA": ("Sanofi", "Europe"),
            "7203.T": ("Toyota", "Asia-Pacific"),
            "6758.T": ("Sony", "Asia-Pacific"),
            "005930.KS": ("Samsung Electronics", "Asia-Pacific"),
            "0700.HK": ("Tencent", "Asia-Pacific"),
            "9988.HK": ("Alibaba", "Asia-Pacific"),
            "RELIANCE.NS": ("Reliance Industries", "Asia-Pacific"),
            "TCS.NS": ("Tata Consultancy", "Asia-Pacific"),
            "HDFCBANK.NS": ("HDFC Bank", "Asia-Pacific"),
            "BHP.AX": ("BHP Group", "Asia-Pacific"),
            "CBA.AX": ("Commonwealth Bank", "Asia-Pacific"),
            "1211.HK": ("BYD", "Asia-Pacific"),
            "7974.T": ("Nintendo", "Asia-Pacific"),
            "INFY.NS": ("Infosys", "Asia-Pacific"),
            "000660.KS": ("SK Hynix", "Asia-Pacific")},
        "benchmark": "^GSPC", "base_currency": "USD"},
    "penny": {          # stocks under 5 (US dollars or Canadian dollars): the list is found by Yahoo's screener every time you rebuild
        "indices": {"^RUT": ("Russell 2000", "North America"), "^GSPC": ("S&P 500", "North America"),
                    "^IXIC": ("Nasdaq Composite", "North America"), "^GSPTSE": ("S&P/TSX Composite", "North America")},
        "stocks": {},    # filled by penny_stocks()
        "benchmark": "^RUT", "base_currency": "USD"},
    "canada": {
        "indices": {
            "^GSPTSE": ("S&P/TSX Composite", "Canada"), "^GSPC": ("S&P 500", "United States"),
            "^IXIC": ("Nasdaq Composite", "United States"), "^DJI": ("Dow Jones", "United States"),
            "^FTSE": ("FTSE 100", "Europe"), "^GDAXI": ("DAX 40", "Europe"), "^N225": ("Nikkei 225", "Asia-Pacific"),
            "^HSI": ("Hang Seng", "Asia-Pacific"), "^AXJO": ("ASX 200", "Asia-Pacific")},
        "stocks": {t: (n, "Canada") for t, n in {
            "RY.TO": "Royal Bank", "TD.TO": "TD Bank", "BNS.TO": "Scotiabank", "BMO.TO": "Bank of Montreal",
            "MFC.TO": "Manulife", "ENB.TO": "Enbridge", "TRP.TO": "TC Energy", "CNQ.TO": "Canadian Natural",
            "SU.TO": "Suncor", "CP.TO": "CPKC", "CNR.TO": "CN Rail", "SHOP.TO": "Shopify", "CSU.TO": "Constellation Software",
            "ATD.TO": "Couche-Tard", "BCE.TO": "BCE", "NTR.TO": "Nutrien", "TECK-B.TO": "Teck Resources",
            "WCN.TO": "Waste Connections", "L.TO": "Loblaw", "FTS.TO": "Fortis"}.items()},
        "benchmark": "^GSPTSE", "base_currency": "CAD"},
}

# ---- CUSTOM: fill these two lists with any Yahoo Finance tickers, then set CONFIG["universe"] = "custom" ----
CUSTOM_UNIVERSE = {
    "indices": ["^GSPTSE", "^GSPC", "^FTSE", "^N225"],
    "stocks":  ["AAPL", "RY.TO", "SHOP.TO", "ASML.AS", "7203.T"],
    "benchmark": "^GSPTSE", "base_currency": "CAD",
}

# region + currency inferred from the Yahoo suffix (used for custom tickers and for FX conversion)
SUFFIX_INFO = {".TO": ("Canada", "CAD", 1), ".V": ("Canada", "CAD", 1), ".L": ("Europe", "GBP", .01), ".DE": ("Europe", "EUR", 1),
               ".PA": ("Europe", "EUR", 1), ".AS": ("Europe", "EUR", 1), ".MI": ("Europe", "EUR", 1), ".MC": ("Europe", "EUR", 1),
               ".SW": ("Europe", "CHF", 1), ".T": ("Asia-Pacific", "JPY", 1), ".HK": ("Asia-Pacific", "HKD", 1),
               ".NS": ("Asia-Pacific", "INR", 1), ".BO": ("Asia-Pacific", "INR", 1), ".AX": ("Asia-Pacific", "AUD", 1),
               ".KS": ("Asia-Pacific", "KRW", 1), ".SS": ("Asia-Pacific", "CNY", 1), ".SZ": ("Asia-Pacific", "CNY", 1),
               ".SA": ("Latin America", "BRL", 1)}
INDEX_CCY = {"^GSPC": "USD", "^IXIC": "USD", "^DJI": "USD", "^GSPTSE": "CAD", "^FTSE": "GBP", "^GDAXI": "EUR", "^FCHI": "EUR",
             "^STOXX50E": "EUR", "^N225": "JPY", "^HSI": "HKD", "^NSEI": "INR", "^AXJO": "AUD", "^KS11": "KRW", "^BVSP": "BRL",
             "000001.SS": "CNY"}

CCY_OVERRIDE = {}          # quote currency chosen by you for tickers the suffix rules cannot identify (set from your holdings)
def ticker_ccy(t):
    # returns (currency, price scale to major units) – e.g. London stocks are quoted in pence
    if t in CCY_OVERRIDE:
        c = CCY_OVERRIDE[t]; return ("GBP", .01) if c == "GBX" else (c, 1.0)
    if t in INDEX_CCY: return INDEX_CCY[t], 1.0
    for suf, (_, ccy, sc) in SUFFIX_INFO.items():
        if t.endswith(suf): return ccy, float(sc)
    return "USD", 1.0

def infer_region(t):
    for suf, (reg, _, _) in SUFFIX_INFO.items():
        if t.endswith(suf): return reg
    return "United States"

PENNY_FALLBACK = {   # used only if Yahoo's screener cannot be reached
    "GRAB": "Grab Holdings Limited",
    "BBD": "Banco Bradesco Sa",
    "OPEN": "Opendoor Technologies Inc",
    "GPRO": "GoPro, Inc.",
    "NIO": "NIO Inc.",
    "HTZ": "Hertz Global Holdings, Inc",
    "ACHR": "Archer Aviation Inc.",
    "AUR": "Aurora Innovation, Inc.",
    "ABEV": "Ambev S.A.",
    "STLA": "Stellantis N.V.",
    "DNN": "Denison Mines Corp",
    "BTG": "B2Gold Corp",
    "JOBY": "Joby Aviation, Inc.",
    "JBLU": "JetBlue Airways Corporation",
    "AMC": "AMC Entertainment Holdings, Inc",
    "TE": "T1 Energy Inc.",
    "RXRX": "Recursion Pharmaceuticals, Inc.",
    "SES": "SES AI Corporation",
    "FRMI": "Fermi Inc.",
    "BBAI": "BigBear.ai, Inc.",
    "SDEV": "Stablecoin Development Corporat",
    "UWMC": "UWM Holdings Corporation",
    "QS": "QuantumScape Corporation",
    "LUMN": "Lumen Technologies, Inc.",
    "GGB": "Gerdau S.A.",
    "KOS": "Kosmos Energy Ltd.",
    "PACB": "Pacific Biosciences of Californ",
    "MPT": "Medical Properties Trust, Inc.",
    "GERN": "Geron Corporation",
    "SBS": "Companhia de saneamento Basico ",
    "LCID": "Lucid Group, Inc.",
    "CHOW": "ChowChow Cloud International Ho",
    "WIT": "Wipro Limited",
    "LAC": "Lithium Americas Corp.",
    "IAUX": "i-80 Gold Corp.",
    "GT": "The Goodyear Tire & Rubber Comp",
    "UAA": "Under Armour, Inc.",
    "INTR": "Inter & Co. Inc.",
    "ENVX": "Enovix Corporation",
    "SID": "Companhia Siderurgica Nacional ",
    "URG": "Ur Energy Inc",
    "SOC": "Sable Offshore Corp.",
    "IQ": "iQIYI, Inc.",
    "BORR": "Borr Drilling Limited",
    "MMA": "MIXED MARTIAL ARTS GROUP LIMITE"}

def penny_stocks(cfg, n_us=140, n_ca=40, max_price=5.0, min_price=0.3):
    """Stocks trading under max_price: the most-traded US names and TSX/TSXV names, found with Yahoo's screener (cached per day)."""
    cache = os.path.join(cfg["cache_dir"], f"penny_{dt.date.today()}.json")
    if cfg.get("use_cache", True) and os.path.exists(cache):
        try: return {t: tuple(v) for t, v in json.load(open(cache, encoding="utf-8")).items()}
        except Exception: pass
    out = {}
    try:
        import yfinance as yf
        from yfinance import EquityQuery as EQ
        def run(exchs, n, min_vol):
            q = EQ("and", [EQ("is-in", ["exchange", *exchs]), EQ("lt", ["intradayprice", max_price]), EQ("gt", ["intradayprice", min_price]), EQ("gt", ["avgdailyvol3m", min_vol])])
            res = yf.screen(q, sortField="avgdailyvol3m", sortAsc=False, size=250).get("quotes", [])
            return [x for x in res if x.get("quoteType") == "EQUITY"][:n]
        for x in run(["NMS", "NYQ", "ASE"], n_us, 1_000_000): out[x["symbol"]] = ((x.get("shortName") or x["symbol"])[:40], "North America")
        for x in run(["TOR", "VAN"], n_ca, 300_000):          out[x["symbol"]] = ((x.get("shortName") or x["symbol"])[:40], "North America")
    except Exception as e:
        print(f"⚠️ penny screener failed ({type(e).__name__}: {e}); using the built-in fallback list")
    if len(out) < 20: out = {t: (n, "North America") for t, n in PENNY_FALLBACK.items()}
    else:
        try: os.makedirs(cfg["cache_dir"], exist_ok=True); json.dump(out, open(cache, "w", encoding="utf-8"))
        except Exception: pass
    return out

def resolve_universe(cfg):
    u = cfg["universe"]
    if u == "custom":
        c = CUSTOM_UNIVERSE
        ind = {t: (t, infer_region(t)) for t in c["indices"]}; stk = {t: (t, infer_region(t)) for t in c["stocks"]}
        return ind, stk, c.get("benchmark") or c["indices"][0], c.get("base_currency", "USD")
    if u not in UNIVERSES: raise ValueError(f"unknown universe '{u}' – use one of {list(UNIVERSES)} or 'custom'")
    p = UNIVERSES[u]
    stocks = penny_stocks(cfg) if u == "penny" else dict(p["stocks"])
    return dict(p["indices"]), stocks, p["benchmark"], p["base_currency"]

INDICES, STOCKS, _bench, _base = resolve_universe(CONFIG)

def load_watchlist(cfg):
    fn = cfg.get("watchlist_csv"); out = []
    if fn and os.path.exists(fn):
        try:
            df = pd.read_csv(fn, dtype=str).fillna(""); df.columns = [c.strip().lower() for c in df.columns]
            for r in df.to_dict("records"):
                t = str(r.get("ticker", "")).strip().upper()
                if t: out.append((t, str(r.get("name", "")).strip(), str(r.get("ccy", "")).strip().upper()))
        except Exception as e:
            print(f"⚠️ could not read {fn}: {e}")
    return out

WATCH = load_watchlist(CONFIG)
_region = lambda t: infer_region(t) if CONFIG["universe"] == "canada" else {"United States": "North America", "Canada": "North America"}.get(infer_region(t), infer_region(t))
for _t, _n, _c in WATCH:                              # stocks you added from the dashboard: they join every universe
    if _t not in INDICES and _t not in STOCKS:
        STOCKS[_t] = (_n or _t, _region(_t))
        if _c: CCY_OVERRIDE[_t] = _c
if WATCH: print(f"Watchlist: {len(WATCH)} extra tickers from {CONFIG['watchlist_csv']}")
BENCH = _bench if CONFIG["benchmark"] == "auto" else CONFIG["benchmark"]
_bc = CONFIG["base_currency"]
BASE_CCY = _base if _bc == "auto" else (None if str(_bc).lower() == "local" else str(_bc).upper())
BASE_LABEL = f"{BASE_CCY}-converted" if BASE_CCY else "local currency"

META = pd.DataFrame(
    [(t, n, r, "Index") for t, (n, r) in INDICES.items()] + [(t, n, r, "Stock") for t, (n, r) in STOCKS.items()],
    columns=["ticker", "name", "region", "type"]).set_index("ticker")
if BENCH not in META.index:                      # allow a benchmark that is not in the lists (e.g. an ETF)
    META.loc[BENCH] = [BENCH, infer_region(BENCH), "Index"]

def load_my_holdings(cfg):
    rows = [dict(h) for h in cfg.get("my_holdings", [])]
    fn = cfg.get("my_holdings_csv")
    if fn and os.path.exists(fn):
        try:
            df = pd.read_csv(fn); df.columns = [c.strip().lower() for c in df.columns]
            df = df.rename(columns={"symbol": "ticker", "qty": "shares", "quantity": "shares", "price": "cost", "avg_cost": "cost", "buy_price": "cost",
                                    "currency": "ccy", "cost_currency": "ccy", "quote_currency": "qccy"})
            rows += [{k: (None if pd.isna(v) else v) for k, v in r.items()} for r in df.to_dict("records")]
        except Exception as e:
            print(f"⚠️ could not read {fn}: {e}")
    _s = lambda v: "" if v is None or str(v).strip().lower() in ("nan", "none") else str(v).strip()
    out = []
    for r in rows:
        t = _s(r.get("ticker")).upper()
        if not t: continue
        cost = _s(r.get("cost"))
        out.append({"t": t, "shares": float(r.get("shares") or 0), "cost": float(cost) if cost else None, "date": _s(r.get("date"))[:10], "note": _s(r.get("note")),
                    "ccy": _s(r.get("ccy")).upper(), "qccy": _s(r.get("qccy")).upper()})
    return out

HOLDINGS = load_my_holdings(CONFIG)
MINE_NEW = list(dict.fromkeys(h["t"] for h in HOLDINGS if h["t"] not in META.index))     # tickers that are not in the universe yet
for h in HOLDINGS:                                    # a quote currency you set for a new ticker overrides the suffix-based guess
    if h["qccy"] and h["t"] in MINE_NEW: CCY_OVERRIDE[h["t"]] = h["qccy"]
for _t in MINE_NEW:
    META.loc[_t] = [_t, infer_region(_t), "Mine"]
ALL_TICKERS = list(META.index)

def label(t):  return f"{META.loc[t, 'name']} ({t})" if t in META.index and META.loc[t, 'name'] != t else t
def short(t):  return META.loc[t, "name"] if t in META.index else t

print(f"My portfolio: {len(HOLDINGS)} holdings ({len(MINE_NEW)} outside the universe)")
print(f"Universe '{CONFIG['universe']}': {len(INDICES)} indices + {len(STOCKS)} stocks = {len(ALL_TICKERS)} instruments | benchmark {BENCH} | analysis currency: {BASE_LABEL}")

## 🔌 Data layer — pluggable providers

Everything downstream only needs a dictionary of wide price tables (`Open, High, Low, Close, Volume` – one column per ticker).
A **provider** is just a function `provider(tickers, cfg) -> dict`. Three are included:

* `yfinance` – free daily data from Yahoo Finance (adjusted for splits and dividends).
* `csv` – your own files (any broker / vendor export).
* `sample` – synthetic, correlated random-walk data so you can try the notebook offline.

**Switching the data source later** → change `CONFIG["data_source"]`, or write your own provider (Alpha Vantage, Polygon, Twelve Data, Bloomberg …) and register it in `PROVIDERS`.

In [ ]:
FIELDS = ["Open", "High", "Low", "Close", "Volume"]

def _naive(df):
    df = df.copy()
    idx = pd.to_datetime(df.index)
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_localize(None)
    df.index = idx.normalize()
    df.index.name = "Date"
    return df[~df.index.duplicated(keep="last")].sort_index()

# ---------- provider 1: Yahoo Finance --------------------------------------------------
def provider_yfinance(tickers, cfg):
    import yfinance as yf
    kw = dict(auto_adjust=True, progress=False, group_by="column", threads=True)
    if cfg["start"]:
        kw.update(start=cfg["start"], end=cfg["end"])
    else:
        kw.update(period=cfg["period"])
    raw = yf.download(list(tickers), **kw)
    if raw is None or raw.empty:
        raise RuntimeError("yfinance returned no data")
    have = set(raw.columns.get_level_values(0))
    return {f: _naive(raw[f]) for f in FIELDS if f in have}

# ---------- provider 2: your own CSV files ---------------------------------------------
def provider_csv(tickers, cfg):
    out = {f: {} for f in FIELDS}
    for t in tickers:
        fn = os.path.join(cfg["csv_folder"], t.replace("^", "_").replace("/", "-") + ".csv")
        if not os.path.exists(fn):
            continue
        df = pd.read_csv(fn)
        df.columns = [c.strip().title() for c in df.columns]
        df["Date"] = pd.to_datetime(df["Date"]);  df = df.set_index("Date").sort_index()
        close = df["Adj Close"] if "Adj Close" in df else df["Close"]
        out["Close"][t] = close
        for f in ["Open", "High", "Low"]:
            out[f][t] = df[f] if f in df else close
        out["Volume"][t] = df["Volume"] if "Volume" in df else pd.Series(np.nan, index=close.index)
    if not out["Close"]:
        raise RuntimeError(f"no CSV files found in {cfg['csv_folder']}")
    return {f: _naive(pd.DataFrame(v)) for f, v in out.items()}

# ---------- provider 3: synthetic demo data --------------------------------------------
def provider_sample(tickers, cfg, seed=7):
    rng = np.random.default_rng(seed)
    p = str(cfg["period"])
    yrs = int(p[:-1]) if p.endswith("y") and p[:-1].isdigit() else 5
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=252 * yrs, name="Date")
    n = len(dates)
    regions = META["region"].reindex(tickers).fillna("Other")
    reg_f = {r: rng.normal(0, .006, n) for r in regions.unique()}
    glob = rng.normal(0.0003, .008, n)
    out = {f: {} for f in FIELDS}
    for t in tickers:
        is_idx = META["type"].get(t, "Stock") == "Index"
        beta = rng.uniform(.8, 1.2) if is_idx else rng.uniform(.6, 1.6)
        r = beta * glob + .7 * reg_f[regions[t]] + rng.normal(0, .004 if is_idx else .014, n)
        close = pd.Series(100 * np.exp(np.cumsum(r)), index=dates)
        rg = np.abs(rng.normal(0, .006 if is_idx else .012, n))
        out["Close"][t], out["High"][t], out["Low"][t] = close, close * (1 + rg), close * (1 - rg)
        out["Open"][t] = close.shift(1).fillna(close.iloc[0])
        out["Volume"][t] = pd.Series(rng.integers(1e6, 5e7, n), index=dates).astype(float)
    return {f: pd.DataFrame(v) for f, v in out.items()}

PROVIDERS = {"yfinance": provider_yfinance, "csv": provider_csv, "sample": provider_sample}
# To add your own:  PROVIDERS["alphavantage"] = my_function   then set CONFIG["data_source"] = "alphavantage"

In [ ]:
DATA_STATUS = {"source": CONFIG["data_source"], "is_sample": CONFIG["data_source"] == "sample", "messages": []}

def load_market_data(tickers, cfg):
    src = cfg["data_source"]
    key = hashlib.md5(json.dumps([sorted(tickers), cfg["period"], cfg["start"], cfg["end"], src],
                                 default=str).encode()).hexdigest()[:10]
    cache_file = os.path.join(cfg["cache_dir"], f"ohlcv_{key}_{dt.date.today()}.pkl")
    if cfg["use_cache"] and src != "sample" and os.path.exists(cache_file):
        DATA_STATUS["messages"].append("Loaded today's cached download.")
        return pd.read_pickle(cache_file)
    try:
        data = PROVIDERS[src](tickers, cfg)
    except Exception as e:
        if not cfg["fallback_to_sample"]:
            raise
        msg = f"⚠️ '{src}' failed ({type(e).__name__}: {e}). Using SYNTHETIC sample data instead."
        print(msg); DATA_STATUS["messages"].append(msg)
        DATA_STATUS.update(is_sample=True, source="sample (fallback)")
        data = provider_sample(tickers, cfg)
    else:
        if cfg["use_cache"] and src != "sample":
            os.makedirs(cfg["cache_dir"], exist_ok=True)
            pd.to_pickle(data, cache_file)
    return data

def clean_market_data(data, min_coverage=0.6, ffill_limit=5):
    close = data["Close"].dropna(how="all")
    close = close[close.index.dayofweek < 5]            # a stray weekend row (a time-zone artefact in one feed) would show as a flat bar and a 0.0% day
    coverage = close.notna().mean()
    dropped = coverage[coverage < min_coverage].index.tolist()
    keep = [c for c in close.columns if c not in dropped]
    out = {}
    for f, df in data.items():
        df = df.reindex(close.index)[keep]
        out[f] = df.ffill(limit=ffill_limit) if f != "Volume" else df
    for f in ["Open", "High", "Low"]:                       # fall back to Close when an OHLC field is missing
        if f in out: out[f] = out[f].fillna(out["Close"])
        else:        out[f] = out["Close"].copy()
    if "Volume" not in out: out["Volume"] = pd.DataFrame(np.nan, index=close.index, columns=keep)
    return out, dropped

raw = load_market_data(ALL_TICKERS, CONFIG)
MKT, dropped = clean_market_data(raw)
close, high, low, volume = MKT["Close"], MKT["High"], MKT["Low"], MKT["Volume"]

close_loc, high_loc, low_loc = close.copy(), high.copy(), low.copy()       # local-currency prices (used for technical indicators)

def load_fx(currencies, cfg, index):
    # units of each currency per 1 USD, from Yahoo pairs like USDCAD=X; forward-filled onto the price calendar
    need = sorted(set(currencies) - {"USD"})
    if not need: return pd.DataFrame(index=index)
    if cfg["data_source"] != "yfinance": raise RuntimeError("FX download needs data_source='yfinance'")
    import yfinance as yf
    cache = os.path.join(cfg["cache_dir"], f"fx_{hashlib.md5(','.join(need).encode()).hexdigest()[:8]}_{cfg['period']}_{cfg['start']}_{dt.date.today()}.pkl")
    if cfg["use_cache"] and os.path.exists(cache): fx = pd.read_pickle(cache)
    else:
        kw = dict(auto_adjust=True, progress=False)
        kw.update(start=cfg["start"], end=cfg["end"]) if cfg["start"] else kw.update(period=cfg["period"])
        fx = _naive(yf.download([f"USD{c}=X" for c in need], **kw)["Close"])
        fx.columns = [c.replace("USD", "", 1).replace("=X", "") for c in fx.columns]
        if cfg["use_cache"]:
            os.makedirs(cfg["cache_dir"], exist_ok=True); fx.to_pickle(cache)
    fx = fx.reindex(fx.index.union(index)).ffill().bfill().reindex(index)
    return fx

FX_NOTE = "no currency conversion (local prices)"
if BASE_CCY:
    try:
        meta = {t: ticker_ccy(t) for t in close.columns}
        fx = load_fx([c for c, _ in meta.values()] + [BASE_CCY], CONFIG, close.index)
        usd_per = lambda c: 1.0 if c == "USD" else 1.0 / fx[c]            # USD value of 1 unit of currency c
        converted, failed = [], []
        for t, (ccy, sc) in meta.items():
            if ccy != "USD" and (ccy not in fx or fx[ccy].isna().all()): failed.append(t); continue
            rate = sc * usd_per(ccy) * (1.0 if BASE_CCY == "USD" else fx[BASE_CCY])         # local price -> BASE_CCY
            if np.isscalar(rate): rate = pd.Series(rate, index=close.index)
            if (t in close.columns): close[t] = close_loc[t] * rate; high[t] = high_loc[t] * rate; low[t] = low_loc[t] * rate
            converted.append(t)
        FX_NOTE = f"all prices converted to {BASE_CCY}" + (f" (not converted: {failed})" if failed else "")
        if failed: DATA_STATUS["messages"].append(f"FX missing for {failed}")
    except Exception as e:
        msg = f"⚠️ FX conversion failed ({type(e).__name__}: {e}) – continuing in local currency."
        print(msg); DATA_STATUS["messages"].append(msg)
        BASE_CCY, BASE_LABEL, FX_NOTE = None, "local currency", "FX conversion failed – local prices"
        close, high, low = close_loc.copy(), high_loc.copy(), low_loc.copy()

DATA_STATUS.update(first=str(close.index[0].date()), last=str(close.index[-1].date()),
                   n_assets=close.shape[1], n_days=len(close), dropped=dropped, currency=BASE_LABEL)
IDX = [t for t in INDICES if t in close.columns]
STK = [t for t in STOCKS if t in close.columns]
MINE = [t for t in MINE_NEW if t in close.columns]
if set(MINE_NEW) - set(MINE): print("⚠️ no price data for your holdings:", sorted(set(MINE_NEW) - set(MINE)))
if BENCH not in close.columns:
    raise ValueError(f"Benchmark {BENCH} has no data – choose another one in CONFIG['benchmark'].")

rets = close.pct_change()
print(f"Source: {DATA_STATUS['source']} | {close.shape[1]} instruments | {len(close):,} trading days | "
      f"{DATA_STATUS['first']} → {DATA_STATUS['last']}")
print("Currency:", FX_NOTE)
if dropped: print("Dropped (not enough data):", dropped)
if DATA_STATUS["is_sample"]: print("⚠️  SAMPLE DATA – numbers are synthetic and meaningless for decisions.")

### Fundamentals loader
Valuation inputs (P/E, P/B, ROE, free-cash-flow …) come from Yahoo's `info` endpoint. They are fetched in parallel and cached for the day. For `csv` mode provide `data/fundamentals.csv` with the columns listed in `FUND_FIELDS.values()`.

In [ ]:
FUND_FIELDS = {"trailingPE": "pe", "forwardPE": "fwd_pe", "priceToBook": "pb", "pegRatio": "peg",
               "returnOnEquity": "roe", "profitMargins": "margin", "earningsGrowth": "eps_growth",
               "revenueGrowth": "rev_growth", "debtToEquity": "debt_equity", "freeCashflow": "fcf",
               "sharesOutstanding": "shares", "marketCap": "mcap", "dividendRate": "div_rate",
               "currency": "currency", "financialCurrency": "fin_currency", "sector": "sector", "industry": "industry", "shortName": "short_name",
               "recommendationMean": "rec_mean", "recommendationKey": "rec_key", "numberOfAnalystOpinions": "n_analysts",
               "targetMeanPrice": "tgt_mean", "targetHighPrice": "tgt_high", "targetLowPrice": "tgt_low", "targetMedianPrice": "tgt_med"}

def _fund_one(t):
    import yfinance as yf
    tk = yf.Ticker(t)
    try:
        info = tk.info or {}
    except Exception:
        info = {}
    out = {v: info.get(k) for k, v in FUND_FIELDS.items()}
    try:                                                        # analyst rating distribution (this month)
        rec = tk.recommendations
        if rec is not None and len(rec):
            for k in ["strongBuy", "buy", "hold", "sell", "strongSell"]:
                out["rec_" + k] = rec.iloc[0].get(k)
    except Exception:
        pass
    return t, out

def sample_fundamentals(tickers, seed=11):
    rng = np.random.default_rng(seed)
    rows = {}
    for t in tickers:
        rows[t] = dict(pe=rng.uniform(8, 45), fwd_pe=rng.uniform(8, 40), pb=rng.uniform(1, 12), peg=rng.uniform(.8, 3),
                       roe=rng.uniform(.05, .35), margin=rng.uniform(.05, .35), eps_growth=rng.uniform(-.1, .4),
                       rev_growth=rng.uniform(-.05, .3), debt_equity=rng.uniform(10, 200), fcf=rng.uniform(1e9, 5e10),
                       shares=rng.uniform(1e9, 1e10), mcap=rng.uniform(1e11, 3e12), div_rate=rng.uniform(0, 4),
                       currency="USD", fin_currency="USD", sector="Sample", industry="Sample",
                       rec_mean=rng.uniform(1.4, 3.2), rec_key="buy", n_analysts=int(rng.integers(8, 40)), tgt_mean=0, tgt_high=0, tgt_low=0, tgt_med=0,
                       rec_strongBuy=int(rng.integers(2, 12)), rec_buy=int(rng.integers(5, 20)), rec_hold=int(rng.integers(2, 10)), rec_sell=int(rng.integers(0, 3)), rec_strongSell=0)
    return pd.DataFrame(rows).T

def load_fundamentals(tickers, cfg):
    cache_file = os.path.join(cfg["cache_dir"], f"fund2_{hashlib.md5(','.join(sorted(tickers)).encode()).hexdigest()[:8]}_{dt.date.today()}.pkl")
    if DATA_STATUS["is_sample"]:
        return sample_fundamentals(tickers)
    if cfg["data_source"] == "csv":
        if os.path.exists(cfg["fundamentals_csv"]):
            return pd.read_csv(cfg["fundamentals_csv"], index_col=0)
        print("No fundamentals CSV found – valuation tables will be empty."); return pd.DataFrame(index=tickers)
    if cfg["use_cache"] and os.path.exists(cache_file):
        return pd.read_pickle(cache_file)
    with ThreadPoolExecutor(8) as ex:
        res = dict(ex.map(_fund_one, tickers))
    df = pd.DataFrame(res).T
    if cfg["use_cache"]:
        os.makedirs(cfg["cache_dir"], exist_ok=True); df.to_pickle(cache_file)
    return df

FUND_RAW = load_fundamentals(STK + MINE, CONFIG)
if "short_name" in FUND_RAW:                                   # nicer names for tickers you added yourself
    for t in FUND_RAW.index:
        if t in META.index and META.loc[t, "name"] == t and isinstance(FUND_RAW.loc[t, "short_name"], str):
            META.loc[t, "name"] = FUND_RAW.loc[t, "short_name"].title()
n_ok = int(FUND_RAW["pe"].notna().sum()) if "pe" in FUND_RAW else 0
print(f"Fundamentals available for {n_ok} of {len(STK)} stocks")

## 🌐 1 · World overview

**Trailing return** over *n* trading days:

$$R_{t,n}=\frac{P_t}{P_{t-n}}-1$$

Horizons used: 1 D, 1 W (5 d), 1 M (21 d), 3 M (63 d), 6 M (126 d), YTD (since last year's final close), 1 Y (252 d).

**Rebased performance** puts every index on the same starting line so growth is directly comparable: $\;100\cdot P_t/P_{0}$.

**Market breadth / regime.** *Breadth* = share of indices trading above their 200-day average. Broad participation (> 66 %) is "risk-on"; narrow (< 33 %) is "risk-off". It is a simple, widely used health check of the global tape.

> **Currency note.** Returns, risk, correlations and portfolios are measured in the **base currency** set in CONFIG (global preset: USD, Canada preset: CAD; `"local"` switches conversion off). A Japanese investor and a Canadian investor see different returns for the same index, and FX moves can add or remove several percent a year. Technical indicators (RSI, MACD, moving averages) stay on **local prices**, because chart signals describe the market itself. Valuation ratios also use local prices.

In [ ]:
HORIZONS = {"1D": 1, "1W": 5, "1M": 21, "3M": 63, "6M": 126, "1Y": 252}

def trailing_returns(px):
    c = px.ffill()
    last = c.iloc[-1]
    out = {k: last / c.shift(n).iloc[-1] - 1 for k, n in HORIZONS.items() if len(c) > n}
    ye = c.loc[:f"{c.index[-1].year - 1}-12-31"]
    out["YTD"] = last / ye.iloc[-1] - 1 if len(ye) else np.nan
    cols = [k for k in ["1D", "1W", "1M", "3M", "6M", "YTD", "1Y"] if k in out]
    return pd.DataFrame(out)[cols]

TR = trailing_returns(close)
world = TR.loc[IDX].join(META[["name", "region"]]).set_index("name", append=False)
display(HTML(f"<h4>Index performance ({BASE_LABEL})</h4>"))
display(world.drop(columns="region").style
        .format("{:+.1%}").background_gradient(cmap="RdYlGn", vmin=-.25, vmax=.25))

In [ ]:
def heatmap_fig(df, title, height=None):
    z = df.values
    txt = [["" if pd.isna(v) else f"{v:+.1%}" for v in row] for row in z]
    fig = go.Figure(go.Heatmap(z=z, x=list(df.columns), y=[label(t) for t in df.index], text=txt,
                               texttemplate="%{text}", colorscale="RdYlGn", zmid=0, zmin=-.3, zmax=.3,
                               colorbar=dict(title="Return", tickformat=".0%"),
                               hovertemplate="%{y}<br>%{x}: %{text}<extra></extra>"))
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(side="top")
    fig.update_layout(title=title, height=height or 60 + 28 * len(df), margin=dict(l=10, r=10, t=90, b=10))
    return fig

heatmap_fig(TR.loc[IDX], f"World indices – trailing returns ({BASE_LABEL})").show()

In [ ]:
def rebased(px, start=None):
    p = px.copy()
    if start: p = p.loc[start:]
    return 100 * p / p.apply(lambda s: s.dropna().iloc[0])

def line_fig(df, title, names=True, height=470, yfmt=None):
    fig = go.Figure()
    for t in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df[t], name=short(t) if names else t, mode="lines",
                                 line=dict(width=1.6), hovertemplate="%{y:.1f}<extra>" + short(t) + "</extra>"))
    fig.update_layout(title=title, height=height, hovermode="x unified", margin=dict(l=10, r=10, t=60, b=10),
                      legend=dict(orientation="h", y=-.15))
    if yfmt: fig.update_yaxes(tickformat=yfmt)
    return fig

line_fig(rebased(close[IDX]), "Indices rebased to 100 (click legend items to hide / isolate)").show()

In [ ]:
def market_regime(close_idx):
    c = close_idx.ffill()
    ma200 = c.rolling(200).mean()
    breadth = float((c.iloc[-1] > ma200.iloc[-1]).mean()) if len(c) >= 200 else np.nan
    r = c.pct_change()
    vol21 = r.rolling(21).std() * np.sqrt(252)
    gvol = vol21.median(axis=1).dropna()
    vol_pct = float((gvol <= gvol.iloc[-1]).mean()) if len(gvol) else np.nan
    regime = ("Risk-on" if breadth > .66 else "Risk-off" if breadth < .33 else "Mixed") if not np.isnan(breadth) else "n/a"
    return dict(breadth=breadth, global_vol=float(gvol.iloc[-1]) if len(gvol) else np.nan, vol_percentile=vol_pct, regime=regime)

REGIME = market_regime(close_loc[IDX])
display(Markdown(
    f"**Regime: {REGIME['regime']}**  ·  breadth (indices above 200-day avg): **{REGIME['breadth']:.0%}**  ·  "
    f"median 21-day volatility: **{REGIME['global_vol']:.1%}** (higher than {REGIME['vol_percentile']:.0%} of its own history)"))

## 📈 2 · Trend & momentum

**Moving averages** smooth noise to reveal direction.

$$\text{SMA}_n(t)=\frac1n\sum_{i=0}^{n-1}P_{t-i}\qquad
\text{EMA}_n(t)=\alpha P_t+(1-\alpha)\,\text{EMA}_n(t-1),\;\; \alpha=\frac{2}{n+1}$$

* **Golden cross**: SMA 50 crosses *above* SMA 200 (long-term uptrend begins). **Death cross**: crosses *below*.
* Price above SMA 200 = long-term uptrend; many professionals simply avoid fighting that line.

**MACD** measures the *gap* between a fast and slow EMA – trend direction and strength:

$$\text{MACD}=\text{EMA}_{12}-\text{EMA}_{26},\quad \text{Signal}=\text{EMA}_9(\text{MACD}),\quad \text{Hist}=\text{MACD}-\text{Signal}$$

Histogram turning positive = upward momentum building; negative = fading.

**RSI** (Wilder, 14) compares average gains to average losses:

$$\text{RSI}=100-\frac{100}{1+RS},\qquad RS=\frac{\text{avg gain}_{14}}{\text{avg loss}_{14}}$$

> 70 = stretched / "overbought" (not an automatic sell!), < 30 = washed-out / "oversold". In strong trends RSI can stay > 70 for months.

**Stochastic %K** shows where the close sits inside the recent high–low range: $\%K=100\cdot\frac{C-L_{14}}{H_{14}-L_{14}}$, with %D a 3-day average of %K.

**Rate of Change** (momentum): $\text{ROC}_n=\frac{P_t}{P_{t-n}}-1$. The classic academic factor is **12-1 momentum**: the return from 12 months ago to 1 month ago (skipping the latest month, which tends to mean-revert).

In [ ]:
def sma(s, n):  return s.rolling(n, min_periods=n).mean()
def ema(s, n):  return s.ewm(span=n, adjust=False, min_periods=n).mean()

def rsi(s, n=14):
    d = s.diff()
    au = d.clip(lower=0).ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    ad = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    out = 100 - 100 / (1 + au / ad.replace(0, np.nan))
    out[(ad == 0) & au.notna()] = 100
    return out

def macd(s, fast=12, slow=26, sig=9):
    m = ema(s, fast) - ema(s, slow)
    sg = m.ewm(span=sig, adjust=False, min_periods=sig).mean()
    return pd.DataFrame({"macd": m, "signal": sg, "hist": m - sg})

def stochastic(h, l, c, k=14, d=3):
    ll, hh = l.rolling(k).min(), h.rolling(k).max()
    K = 100 * (c - ll) / (hh - ll).replace(0, np.nan)
    return pd.DataFrame({"%K": K, "%D": K.rolling(d).mean()})

def roc(s, n):  return s / s.shift(n) - 1

def atr(h, l, c, n=14):
    pc = c.shift()
    tr = pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()

def bollinger(s, n=20, k=2):
    m, sd = s.rolling(n).mean(), s.rolling(n).std(ddof=0)
    up, lo = m + k * sd, m - k * sd
    return pd.DataFrame({"mid": m, "upper": up, "lower": lo, "pct_b": (s - lo) / (up - lo), "bandwidth": (up - lo) / m})

def ewma_vol(r, lam=0.94, ann=252):                    # RiskMetrics exponentially-weighted volatility
    return np.sqrt((r ** 2).ewm(alpha=1 - lam, adjust=False).mean() * ann)

def ohlc(t):                                           # technical analysis always uses LOCAL prices
    c = close_loc[t].dropna()
    return dict(c=c, h=high_loc[t].reindex(c.index).fillna(c), l=low_loc[t].reindex(c.index).fillna(c))

def indicator_frame(t):
    o = ohlc(t); c = o["c"]
    df = pd.DataFrame({"close": c, "sma20": sma(c, 20), "sma50": sma(c, 50), "sma200": sma(c, 200), "ema20": ema(c, 20)})
    df["rsi"] = rsi(c)
    df = df.join(macd(c)).join(stochastic(o["h"], o["l"], c)).join(bollinger(c).add_prefix("bb_"))
    df["atr"] = atr(o["h"], o["l"], c)
    return df
print("Indicator library ready.")

In [ ]:
def plot_trend(t, last_n=None):
    df = indicator_frame(t)
    if last_n: df = df.iloc[-last_n:]
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=.03, row_heights=[.46, .18, .18, .18],
                        subplot_titles=("Price with moving averages", "RSI (14)", "MACD (12, 26, 9)", "Stochastic (14, 3)"))
    for col, color, w in [("close", "#111827", 1.6), ("sma50", "#2563eb", 1.2), ("sma200", "#dc2626", 1.4), ("ema20", "#d97706", 1)]:
        fig.add_trace(go.Scatter(x=df.index, y=df[col], name=col.upper(), line=dict(color=color, width=w)), 1, 1)
    # golden / death cross markers
    state = (df["sma50"] > df["sma200"]).where(df["sma200"].notna())
    flip = state.astype(float).diff()
    for val, nm, col, sym in [(1, "Golden cross", "#059669", "triangle-up"), (-1, "Death cross", "#dc2626", "triangle-down")]:
        pts = df.index[flip == val]
        fig.add_trace(go.Scatter(x=pts, y=df.loc[pts, "sma200"], mode="markers", name=nm,
                                 marker=dict(symbol=sym, size=13, color=col, line=dict(width=1, color="white"))), 1, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df["rsi"], name="RSI", line=dict(color="#7c3aed", width=1.3)), 2, 1)
    for lvl in (30, 70): fig.add_hline(y=lvl, line=dict(color="#9ca3af", dash="dot"), row=2, col=1)
    fig.add_trace(go.Bar(x=df.index, y=df["hist"], name="Histogram",
                         marker_color=np.where(df["hist"] >= 0, "#34d399", "#f87171")), 3, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df["macd"], name="MACD", line=dict(color="#2563eb", width=1.2)), 3, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df["signal"], name="Signal", line=dict(color="#d97706", width=1.2)), 3, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df["%K"], name="%K", line=dict(color="#0891b2", width=1)), 4, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df["%D"], name="%D", line=dict(color="#db2777", width=1)), 4, 1)
    for lvl in (20, 80): fig.add_hline(y=lvl, line=dict(color="#9ca3af", dash="dot"), row=4, col=1)
    fig.update_layout(height=820, title=f"Trend & momentum – {label(t)} (local currency)", hovermode="x unified",
                      margin=dict(l=10, r=10, t=80, b=10), legend=dict(orientation="h", y=-.06))
    return fig

FOCUS = CONFIG["focus_ticker"] if CONFIG["focus_ticker"] in close.columns else STK[0]
plot_trend(FOCUS).show()

### Signal table — one row per instrument
Five transparent yes/no tests are combined into a **trend score (0–5)**:
1. Price > SMA 200 · 2. SMA 50 > SMA 200 · 3. MACD histogram > 0 · 4. RSI between 50 and 70 (healthy strength, not stretched) · 5. 12-1 momentum > 0.

Read it as a *screen*, not a verdict: a score of 5 means "everything is pointing up", 0 means "everything is pointing down".

In [ ]:
def signal_table(tickers):
    rows = {}
    for t in tickers:
        df = indicator_frame(t).dropna(subset=["sma200"])
        if df.empty: continue
        last = df.iloc[-1]; c = close[t].dropna()           # momentum measured in analysis currency
        m121 = c.iloc[-22] / c.iloc[-253] - 1 if len(c) > 253 else np.nan
        tests = [last["close"] > last["sma200"], last["sma50"] > last["sma200"], last["hist"] > 0, 50 <= last["rsi"] <= 70,
                 (m121 > 0) if not np.isnan(m121) else False]
        rows[t] = {"Name": short(t), "Price vs SMA200": last["close"] / last["sma200"] - 1, "RSI": last["rsi"],
                   "MACD hist %": last["hist"] / last["close"], "Stoch %K": last["%K"], "ROC 3M": roc(c, 63).iloc[-1], "Mom 12-1": m121,
                   "Cross state": "Golden" if last["sma50"] > last["sma200"] else "Death", "Trend score": int(sum(tests))}
    return pd.DataFrame(rows).T

SIG = signal_table(IDX + STK + MINE)
num = ["Price vs SMA200", "RSI", "MACD hist %", "Stoch %K", "ROC 3M", "Mom 12-1"]
SIG[num] = SIG[num].astype(float)
display(SIG.sort_values("Trend score", ascending=False).style
        .format({"Price vs SMA200": "{:+.1%}", "RSI": "{:.0f}", "MACD hist %": "{:+.2%}", "Stoch %K": "{:.0f}",
                 "ROC 3M": "{:+.1%}", "Mom 12-1": "{:+.1%}"})
        .background_gradient(cmap="RdYlGn", subset=["Trend score"], vmin=0, vmax=5)
        .background_gradient(cmap="RdYlGn", subset=["Mom 12-1"], vmin=-.4, vmax=.6))

In [ ]:
# Static chart (matplotlib): 12-1 momentum league table
m = SIG["Mom 12-1"].astype(float).dropna().sort_values()
fig, ax = plt.subplots(figsize=(10, max(4, .27 * len(m))))
ax.barh([short(t) for t in m.index], m.values, color=np.where(m.values >= 0, "#10b981", "#ef4444"))
ax.set_title("12-1 month momentum (return from 12 months ago to 1 month ago)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.axvline(0, color="k", lw=.8)
plt.tight_layout(); plt.show()

## 🌊 3 · Volatility & risk

**Volatility** is the annualised standard deviation of returns: $\sigma_{ann}=\sigma_{daily}\sqrt{252}$.
**EWMA volatility** (RiskMetrics, $\lambda=0.94$) reacts faster to shocks: $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_t^2$.

**Bollinger Bands**: SMA 20 ± 2 standard deviations. $\%B=\frac{P-\text{Lower}}{\text{Upper}-\text{Lower}}$ (0 = at lower band, 1 = at upper). *Bandwidth* squeezes → volatility compression, often before a larger move.

**ATR** (Average True Range) is volatility in *price units*: $TR=\max(H-L,\;|H-C_{t-1}|,\;|L-C_{t-1}|)$, smoothed over 14 days. Handy for position sizing and stop distances.

### Risk-adjusted performance
| Metric | Formula | Reads as |
|---|---|---|
| **CAGR** | $(P_{end}/P_{start})^{1/years}-1$ | Smoothed yearly growth |
| **Sharpe** | $\dfrac{\bar r-r_f}{\sigma}$ | Excess return per unit of *total* risk. > 1 good, > 2 exceptional |
| **Sortino** | $\dfrac{\bar r-r_f}{\sigma_{down}}$ | Same, but penalises only *downside* volatility |
| **Max drawdown** | $\min_t\left(\dfrac{P_t}{\max_{s\le t}P_s}-1\right)$ | Worst peak-to-trough loss you would have lived through |
| **Calmar** | $\text{CAGR}/\lvert\text{MaxDD}\rvert$ | Return earned per unit of worst-case pain |

### Value-at-Risk (VaR) and Expected Shortfall (CVaR)
*"With 95 % confidence a one-day loss will not exceed VaR."* Three estimators are shown:
* **Historical** – the 5th percentile of actual returns (no distribution assumption).
* **Parametric (Normal)** – $\text{VaR}=-(\mu+z_\alpha\sigma)$; assumes a bell curve, so it *underestimates* fat tails.
* **Cornish-Fisher** – corrects $z_\alpha$ for skewness $S$ and excess kurtosis $K$:
$$z_{cf}=z+\tfrac{(z^2-1)S}{6}+\tfrac{(z^3-3z)K}{24}-\tfrac{(2z^3-5z)S^2}{36}$$
* **CVaR / Expected Shortfall** – the *average* loss on the days that are worse than VaR. It answers "if it's bad, how bad?" and is preferred by modern regulation.

In [ ]:
ANN = CONFIG["trading_days"]; RF = CONFIG["risk_free_rate"]; CONF = CONFIG["confidence"]

def drawdown(p):  return p / p.cummax() - 1
def cagr(p):
    p = p.dropna(); yrs = (p.index[-1] - p.index[0]).days / 365.25
    return (p.iloc[-1] / p.iloc[0]) ** (1 / yrs) - 1 if yrs > 0 else np.nan

def risk_metrics(px, rf=RF, conf=CONF, ann=ANN):
    rows = {}
    for t in px.columns:
        p = px[t].dropna(); r = p.pct_change().dropna()
        if len(r) < 60: continue
        rf_d = rf / ann; sd = r.std() * np.sqrt(ann)
        down = np.sqrt((np.minimum(r - rf_d, 0) ** 2).mean()) * np.sqrt(ann)
        dd = drawdown(p); uw = dd < 0
        a = 1 - conf; z = stats.norm.ppf(a); S = stats.skew(r); K = stats.kurtosis(r)
        zcf = z + (z**2 - 1) * S / 6 + (z**3 - 3 * z) * K / 24 - (2 * z**3 - 5 * z) * S**2 / 36
        q = np.quantile(r, a)
        rows[t] = {"CAGR": cagr(p), "Volatility": sd, "Sharpe": (r.mean() - rf_d) * ann / sd,
                   "Sortino": (r.mean() - rf_d) * ann / down, "Max DD": dd.min(), "Calmar": cagr(p) / abs(dd.min()),
                   "Days under water (max)": int(uw.groupby((~uw).cumsum()).cumsum().max()),
                   "VaR hist": -q, "VaR normal": -(r.mean() + z * r.std()), "VaR Cornish-Fisher": -(r.mean() + zcf * r.std()),
                   "CVaR": -r[r <= q].mean(), "Skew": S, "Excess kurt": K, "Worst day": r.min(), "Best day": r.max()}
    return pd.DataFrame(rows).T.astype(float)

RISK = risk_metrics(close)
RISK["Name"] = [short(t) for t in RISK.index]
cols = ["Name", "CAGR", "Volatility", "Sharpe", "Sortino", "Max DD", "Calmar", "VaR hist", "VaR normal",
        "VaR Cornish-Fisher", "CVaR", "Skew", "Excess kurt", "Worst day"]
fmt = {c: "{:+.1%}" for c in ["CAGR", "Max DD", "Worst day"]}
fmt.update({c: "{:.1%}" for c in ["Volatility", "VaR hist", "VaR normal", "VaR Cornish-Fisher", "CVaR"]})
fmt.update({c: "{:.2f}" for c in ["Sharpe", "Sortino", "Calmar", "Skew", "Excess kurt"]})
display(HTML(f"<h4>Risk & return — daily data, {CONF:.0%} one-day VaR, risk-free {RF:.1%}</h4>"))
display(RISK.loc[IDX + STK, cols].style.format(fmt)
        .background_gradient(cmap="RdYlGn", subset=["Sharpe", "Sortino", "Calmar"])
        .background_gradient(cmap="RdYlGn_r", subset=["Volatility", "VaR hist", "CVaR"]))

In [ ]:
def plot_vol_bands(t, last_n=500):
    df = indicator_frame(t).iloc[-last_n:]
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=.04, row_heights=[.55, .22, .23],
                        subplot_titles=("Bollinger Bands (20, 2)", "Bandwidth  (squeeze = low volatility)", "ATR (14) as % of price"))
    fig.add_trace(go.Scatter(x=df.index, y=df.bb_upper, line=dict(width=0), showlegend=False, hoverinfo="skip"), 1, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df.bb_lower, fill="tonexty", fillcolor="rgba(37,99,235,.12)",
                             line=dict(width=0), name="Band", hoverinfo="skip"), 1, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df.bb_mid, name="SMA 20", line=dict(color="#2563eb", width=1)), 1, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df.close, name="Close", line=dict(color="#111827", width=1.5)), 1, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df.bb_bandwidth, name="Bandwidth", line=dict(color="#7c3aed")), 2, 1)
    fig.add_trace(go.Scatter(x=df.index, y=df.atr / df.close, name="ATR %", line=dict(color="#d97706")), 3, 1)
    fig.update_yaxes(tickformat=".0%", row=2, col=1); fig.update_yaxes(tickformat=".1%", row=3, col=1)
    fig.update_layout(height=700, title=f"Volatility toolkit – {label(t)} (local currency)", hovermode="x unified",
                      margin=dict(l=10, r=10, t=80, b=10), showlegend=False)
    return fig

plot_vol_bands(FOCUS).show()

In [ ]:
r_idx = rets[IDX]
vol_fig = go.Figure()
for t in IDX:
    vol_fig.add_trace(go.Scatter(x=r_idx.index, y=r_idx[t].rolling(63).std() * np.sqrt(ANN), name=short(t), line=dict(width=1.3)))
vol_fig.add_trace(go.Scatter(x=r_idx.index, y=ewma_vol(rets[BENCH].fillna(0)), name=f"{short(BENCH)} – EWMA(0.94)",
                             line=dict(color="black", width=2.5, dash="dot")))
vol_fig.update_layout(title="Rolling 63-day annualised volatility (black dotted = EWMA vol of the benchmark)",
                      height=460, hovermode="x unified", yaxis_tickformat=".0%", legend=dict(orientation="h", y=-.15),
                      margin=dict(l=10, r=10, t=60, b=10))
vol_fig.show()

In [ ]:
# Static charts (matplotlib): return distribution with VaR/CVaR, and drawdowns
r = rets[FOCUS].dropna()
a = 1 - CONF; z = stats.norm.ppf(a)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hist(r, bins=70, density=True, color="#93c5fd", edgecolor="white")
xs = np.linspace(r.min(), r.max(), 300)
ax[0].plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), color="#1d4ed8", lw=1.6, label="Normal fit")
ax[0].axvline(-RISK.loc[FOCUS, "VaR hist"], color="#dc2626", ls="--", label=f"VaR {CONF:.0%} = {RISK.loc[FOCUS,'VaR hist']:.1%}")
ax[0].axvline(-RISK.loc[FOCUS, "CVaR"], color="#7f1d1d", ls="-", label=f"CVaR = {RISK.loc[FOCUS,'CVaR']:.1%}")
ax[0].set_title(f"{short(FOCUS)} – daily return distribution (fat tails vs the normal curve)")
ax[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax[0].legend()
for t in [FOCUS, BENCH]:
    ax[1].fill_between(close.index, drawdown(close[t].dropna()).reindex(close.index), 0, alpha=.35, label=short(t))
ax[1].set_title("Drawdown from previous peak"); ax[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax[1].legend()
plt.tight_layout(); plt.show()

## 🔗 4 · Cross-market analysis

### Correlation & PCA — *how synchronised is the world?*
$$\rho_{ij}=\frac{\operatorname{Cov}(r_i,r_j)}{\sigma_i\sigma_j}$$

* ρ = +1 move together, 0 unrelated, −1 opposite. Diversification only helps when correlations are **below 1**.
* The matrix is **re-ordered by hierarchical clustering** (distance $d_{ij}=\sqrt{\tfrac12(1-\rho_{ij})}$) so related markets sit next to each other and blocks become visible.
* **PCA**: the share of total variance explained by the first principal component is the strength of the single "global factor". A high number (> 50 %) means markets mostly move together, so diversification is weaker.

> **Why weekly data?** Tokyo closes *before* New York opens, so a daily Nikkei move and S&P move are measured on different clocks and look artificially unrelated (non-synchronous trading). Weekly returns largely remove that bias. Set `cross_market_freq="D"` to compare.

### CAPM statistics versus the benchmark
Regress excess returns of an asset on excess returns of the benchmark: $\;r_i-r_f=\alpha+\beta\,(r_b-r_f)+\varepsilon$

| Statistic | Meaning |
|---|---|
| **Beta β** | Sensitivity to the benchmark. β = 1.3 → typically moves 30 % more than the market |
| **Alpha α** (annualised) | Return *not* explained by the market. Check the **t-stat**: \|t\| < 2 means it could easily be luck |
| **R²** | Share of the asset's variance explained by the benchmark |
| **Tracking error** | Std-dev of $(r_i-r_b)$, annualised |
| **Information ratio** | Average active return ÷ tracking error: consistency of beating the benchmark |
| **Up / Down capture** | How much of the benchmark's up (down) moves the asset captured. Up > 100 % and Down < 100 % is ideal |

In [ ]:
XF = CONFIG["cross_market_freq"]
if XF == "D":
    xclose, ANN_X, XWIN = close, ANN, 252
else:
    xclose, ANN_X, XWIN = close.resample(XF).last(), 52, 52
xr = xclose.pct_change()

def cluster_order(corr):
    d = np.sqrt(0.5 * (1 - corr.clip(-1, 1).fillna(0))).values.copy()
    np.fill_diagonal(d, 0)
    link = sch.linkage(squareform(d, checks=False), "average")
    return corr.index[sch.leaves_list(link)]

def corr_fig(tickers, title):
    c = xr[tickers].corr()
    order = cluster_order(c)
    c = c.loc[order, order]
    names = [short(t) for t in order]
    fig = go.Figure(go.Heatmap(z=c.values, x=names, y=names, zmin=-1, zmax=1, colorscale="RdBu_r", zmid=0,
                               text=np.round(c.values, 2), texttemplate="%{text}", textfont=dict(size=9),
                               colorbar=dict(title="ρ")))
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title=title, height=640, margin=dict(l=10, r=10, t=60, b=10))
    return fig, c

fig_corr_idx, CORR_IDX = corr_fig(IDX, f"Correlation of index returns ({'weekly' if XF != 'D' else 'daily'}), clustered")
fig_corr_idx.show()

ev = np.sort(np.linalg.eigvalsh(xr[IDX].dropna().corr().values))[::-1]
PC_SHARE = ev / ev.sum()
avg_corr = CORR_IDX.values[np.triu_indices(len(CORR_IDX), 1)].mean()
display(Markdown(f"**Average pairwise correlation of indices: {avg_corr:.2f}** · first principal component explains **{PC_SHARE[0]:.0%}** "
                 f"of all index variance (PC2 {PC_SHARE[1]:.0%}, PC3 {PC_SHARE[2]:.0%}) → the 'global factor' is "
                 f"{'strong' if PC_SHARE[0] > .5 else 'moderate' if PC_SHARE[0] > .35 else 'weak'}."))

In [ ]:
fig_corr_stk, CORR_STK = corr_fig(STK, "Correlation of stock returns, clustered")
fig_corr_stk.show()

roll = xr[IDX].drop(columns=BENCH, errors="ignore").rolling(XWIN).corr(xr[BENCH])
fig_roll = line_fig(roll, f"Rolling 1-year correlation with {short(BENCH)} – correlations rise in crises", height=430)
fig_roll.update_yaxes(range=[-.5, 1]); fig_roll.show()

In [ ]:
def capm_stats(returns, bench_ret, rf=RF, ann=ANN_X):
    rows = {}; rfp = rf / ann
    for t in returns.columns:
        if t == bench_ret.name: continue
        d = pd.concat([returns[t], bench_ret], axis=1).dropna()
        if len(d) < 30: continue
        y, x = d.iloc[:, 0], d.iloc[:, 1]
        res = stats.linregress(x - rfp, y - rfp)
        te = (y - x).std() * np.sqrt(ann); up = x > 0
        rows[t] = {"Beta": res.slope, "Alpha (ann.)": res.intercept * ann, "Alpha t-stat": res.intercept / res.intercept_stderr,
                   "R²": res.rvalue ** 2, "Correlation": res.rvalue, "Tracking error": te,
                   "Info ratio": (y - x).mean() * ann / te,
                   "Up capture": y[up].mean() / x[up].mean(), "Down capture": y[~up].mean() / x[~up].mean()}
    return pd.DataFrame(rows).T.astype(float)

CAPM = capm_stats(xr, xr[BENCH])
CAPM.insert(0, "Name", [short(t) for t in CAPM.index])
f = {"Beta": "{:.2f}", "Alpha (ann.)": "{:+.1%}", "Alpha t-stat": "{:+.1f}", "R²": "{:.0%}", "Correlation": "{:.2f}",
     "Tracking error": "{:.1%}", "Info ratio": "{:+.2f}", "Up capture": "{:.0%}", "Down capture": "{:.0%}"}
display(HTML(f"<h4>CAPM statistics vs {label(BENCH)}  ({'weekly' if XF!='D' else 'daily'} returns, risk-free {RF:.1%})</h4>"))
display(CAPM.loc[[t for t in IDX + STK if t in CAPM.index]].style.format(f)
        .background_gradient(cmap="RdYlGn", subset=["Alpha (ann.)", "Info ratio"])
        .background_gradient(cmap="Blues", subset=["Beta"]))

In [ ]:
# Relative strength: asset / benchmark.  Rising line = outperforming.
rs = close[STK].div(close[BENCH], axis=0)
rs = 100 * rs / rs.apply(lambda s: s.dropna().iloc[0])
line_fig(rs, f"Relative strength vs {short(BENCH)} (rebased to 100; rising = outperforming)", height=480).show()

## 🧮 5 · Portfolio construction — Modern Portfolio Theory

For weights $w$ with expected returns $\mu$ and covariance $\Sigma$:

$$\mu_p=w^\top\mu,\qquad \sigma_p=\sqrt{w^\top\Sigma\,w},\qquad \text{Sharpe}=\frac{\mu_p-r_f}{\sigma_p}$$

* The **efficient frontier** is the set of portfolios giving the *highest return for each level of risk*.
* **Min-variance** portfolio = the very left tip of the frontier. **Max-Sharpe** = the point where a line from the risk-free rate just touches the frontier (best return per unit of risk).
* The grey cloud is thousands of random portfolios: everything efficient lies on its upper-left edge.

**Why this is usually *too* good on paper — and what we do about it.** Optimisers amplify estimation error: five years of NVIDIA's great run would make it 100 % of the "optimal" portfolio. Two professional safeguards are applied (set both in CONFIG):
1. **Weight cap** (`max_weight`, default 25 %) so no single stock dominates.
2. **Shrinkage**: expected returns are pulled toward the group average (`mu_shrinkage`) and covariances toward their diagonal (`cov_shrinkage`).

Stocks are in different currencies and trading hours, so weekly local-currency returns are used. Treat the output as a *framework for thinking about diversification*, not an instruction.

In [ ]:
def ef_inputs(tickers, returns, ann, mu_sh, cov_sh):
    R = returns[tickers].dropna()
    mu = R.mean() * ann
    mu = (1 - mu_sh) * mu + mu_sh * mu.mean()
    S = R.cov() * ann
    S = (1 - cov_sh) * S + cov_sh * np.diag(np.diag(S))
    return mu.values, S.values, R

def optimise(mu, S, rf, max_w, kind="sharpe", target=None):
    n = len(mu); w0 = np.full(n, 1 / n)
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
    if kind == "sharpe":   obj = lambda w: -(w @ mu - rf) / np.sqrt(w @ S @ w)
    elif kind == "minvar": obj = lambda w: w @ S @ w
    else:
        cons.append({"type": "eq", "fun": lambda w: w @ mu - target}); obj = lambda w: w @ S @ w
    res = optimize.minimize(obj, w0, bounds=[(0, max_w)] * n, constraints=cons, method="SLSQP", options={"maxiter": 600})
    return res.x if res.success else None

def pstats(w, mu, S, rf):
    ret, vol = float(w @ mu), float(np.sqrt(w @ S @ w))
    return ret, vol, (ret - rf) / vol

PF_T = [t for t in STK if xr[t].notna().sum() > 60]
mu, S, R = ef_inputs(PF_T, xr, ANN_X, CONFIG["mu_shrinkage"], CONFIG["cov_shrinkage"])
n = len(PF_T); max_w = max(CONFIG["max_weight"], 1 / n + 1e-6)

w_sharpe = optimise(mu, S, RF, max_w, "sharpe")
w_minvar = optimise(mu, S, RF, max_w, "minvar")
w_equal = np.full(n, 1 / n)

# frontier curve: from min-variance return up to the greedy maximum-return portfolio under the cap
w_maxret = np.zeros(n); left = 1.0
for i in np.argsort(-mu):
    w_maxret[i] = min(max_w, left); left -= w_maxret[i]
    if left <= 1e-12: break
targets = np.linspace(pstats(w_minvar, mu, S, RF)[0], pstats(w_maxret, mu, S, RF)[0] * .999, 35)
frontier, frontier_w = [], []
for tg in targets:
    w = optimise(mu, S, RF, max_w, "target", tg)
    if w is not None:
        frontier.append(pstats(w, mu, S, RF)[:2]); frontier_w.append(w)
frontier = np.array(frontier)

rng = np.random.default_rng(1)
W = rng.dirichlet(np.ones(n), 40000)
W = W[W.max(axis=1) <= max_w][:CONFIG["n_random_portfolios"]]
cloud = np.array([pstats(w, mu, S, RF) for w in W])

PORT = pd.DataFrame({"Max Sharpe": w_sharpe, "Min variance": w_minvar, "Equal weight": w_equal}, index=PF_T)
PORT_STATS = pd.DataFrame({k: pstats(PORT[k].values, mu, S, RF) for k in PORT}, index=["Exp. return", "Volatility", "Sharpe"]).T
display(PORT_STATS.style.format({"Exp. return": "{:.1%}", "Volatility": "{:.1%}", "Sharpe": "{:.2f}"}))
print(f"Random portfolios plotted: {len(cloud)}  |  assets: {n}  |  weight cap: {max_w:.0%}")

In [ ]:
def frontier_fig():
    fig = go.Figure()
    if len(cloud):
        fig.add_trace(go.Scatter(x=cloud[:, 1], y=cloud[:, 0], mode="markers", name="Random portfolios",
                                 marker=dict(size=4, color=cloud[:, 2], colorscale="Viridis", opacity=.55,
                                             colorbar=dict(title="Sharpe")), hovertemplate="Vol %{x:.1%}<br>Return %{y:.1%}<extra></extra>"))
    fig.add_trace(go.Scatter(x=frontier[:, 1], y=frontier[:, 0], mode="lines", name="Efficient frontier", line=dict(color="#dc2626", width=3)))
    for nm, col, sym in [("Max Sharpe", "#f59e0b", "star"), ("Min variance", "#2563eb", "diamond"), ("Equal weight", "#111827", "circle")]:
        rr, vv, ss = PORT_STATS.loc[nm]
        fig.add_trace(go.Scatter(x=[vv], y=[rr], mode="markers+text", name=nm, text=[nm], textposition="top center",
                                 marker=dict(size=15, color=col, symbol=sym, line=dict(width=1, color="white"))))
    sv = np.sqrt(np.diag(S))
    fig.add_trace(go.Scatter(x=sv, y=mu, mode="markers+text", text=[t.split(".")[0] for t in PF_T], textposition="bottom center",
                             textfont=dict(size=9), name="Individual stocks", marker=dict(size=7, color="#6b7280")))
    fig.update_layout(title="Efficient frontier (shrunk inputs, weight-capped)", height=560, xaxis_title="Volatility (annual)",
                      yaxis_title="Expected return (annual)", xaxis_tickformat=".0%", yaxis_tickformat=".0%",
                      legend=dict(orientation="h", y=-.18), margin=dict(l=10, r=10, t=60, b=10))
    return fig

def weights_fig():
    fig = go.Figure()
    for k, col in [("Max Sharpe", "#f59e0b"), ("Min variance", "#2563eb"), ("Equal weight", "#9ca3af")]:
        fig.add_trace(go.Bar(x=[short(t) for t in PORT.index], y=PORT[k], name=k, marker_color=col))
    fig.update_layout(barmode="group", title="Portfolio weights", height=420, yaxis_tickformat=".0%",
                      legend=dict(orientation="h", y=-.3), margin=dict(l=10, r=10, t=60, b=10))
    return fig

frontier_fig().show(); weights_fig().show()

## 💼 6 · Valuation & fundamentals

| Metric | Formula | Rule of thumb |
|---|---|---|
| **P/E** | Price ÷ earnings per share | Low = cheap vs profits; very high = growth priced in |
| **Earnings yield** | 1 ÷ P/E | Compare directly with bond yields |
| **P/B** | Price ÷ book value per share | < 1 trades below accounting value |
| **FCF yield** | Free cash flow ÷ market cap | Cash actually generated per dollar invested |
| **Dividend yield** | Annual dividend ÷ price | Cash paid to you |
| **ROE** | Net income ÷ equity | Quality of the business: > 15 % is solid |
| **Net margin** | Net income ÷ revenue | Pricing power |
| **Debt/Equity** | Total debt ÷ equity | Balance-sheet leverage |
| **PEG** | P/E ÷ earnings growth | ≈ 1 fair, > 2 expensive for the growth |

### Simple DCF (discounted cash flow)
Value today = present value of future free cash flow (FCF) plus a terminal value:

$$V_0=\sum_{t=1}^{5}\frac{FCF_0(1+g)^t}{(1+r)^t}+\frac{FCF_5\,(1+g_T)}{(r-g_T)\,(1+r)^5},\qquad \text{per share}=\frac{V_0}{\text{shares}}$$

$g$ = 5-year growth, $g_T$ = terminal growth, $r$ = discount rate. The result is **extremely sensitive** to these inputs, which is why a sensitivity grid is shown. A DCF is only computed when the company reports in the same currency its shares trade in (otherwise the comparison would be silently wrong). **Caveat:** trailing free cash flow can be temporarily depressed by heavy investment (e.g. AI data-centre capex), which makes a 5-year DCF look very pessimistic for those names – use the sensitivity grid to see what growth the current price implies.

### The 4-factor scorecard
Each stock gets a **z-score** per metric, $z=\frac{x-\bar x}{s}$ (winsorised at ±3 so outliers can't dominate), averaged into four factors – the same families used in academic factor investing:

* **Value** – earnings yield, book yield (1/PB), FCF yield, dividend yield
* **Quality** – ROE (capped at 60 %), net margin, low debt/equity
* **Momentum** – 12-1 month return, 6-month return, distance above SMA 200
* **Risk (low-vol)** – low volatility, shallow max drawdown, high Sharpe

The weighted composite (see `scorecard_weights`) becomes a 0–100 percentile rank. It ranks stocks **relative to each other within this list** – it does not say any stock is a good buy in absolute terms.

In [ ]:
def build_fund_table(raw_fund, px):
    df = raw_fund.copy()
    for c in ["pe", "fwd_pe", "pb", "peg", "roe", "margin", "eps_growth", "rev_growth", "debt_equity", "fcf", "shares", "mcap", "div_rate",
              "rec_mean", "n_analysts", "tgt_mean", "tgt_high", "tgt_low", "tgt_med", "rec_strongBuy", "rec_buy", "rec_hold", "rec_sell", "rec_strongSell"]:
        df[c] = pd.to_numeric(df[c] if c in df else pd.Series(np.nan, index=df.index), errors="coerce")
    df["price"] = px.ffill().iloc[-1].reindex(df.index)
    if DATA_STATUS["is_sample"]:                               # synthetic analyst targets around the synthetic price
        _r = np.random.default_rng(5); df["tgt_mean"] = df["price"] * _r.uniform(.9, 1.35, len(df)); df["tgt_high"] = df["tgt_mean"] * 1.25; df["tgt_low"] = df["tgt_mean"] * .8; df["tgt_med"] = df["tgt_mean"]
    major = df["price"].where(~df["currency"].astype(str).isin(["GBp", "GBX", "ZAc", "ILA"]), df["price"] / 100)   # pence -> pounds
    df["div_yield"] = df["div_rate"] / major
    df.loc[df["div_yield"] > .20, "div_yield"] = np.nan             # quote-unit mismatch guard (e.g. pence vs pounds)
    df["debt_equity"] = df["debt_equity"] / 100                    # Yahoo reports percent
    df["earn_yield"] = 1 / df["pe"].where(df["pe"] > 0)
    df["book_yield"] = 1 / df["pb"].where(df["pb"] > 0)
    same = df["currency"].astype(str) == df["fin_currency"].astype(str)
    df["same_ccy"] = same
    df["fcf_yield"] = (df["fcf"] / df["mcap"]).where(same)
    df.insert(0, "name", [short(t) for t in df.index])
    return df

FUND = build_fund_table(FUND_RAW.reindex(STK), close_loc)      # valuation ratios use local prices
FUND_MINE = build_fund_table(FUND_RAW.reindex(MINE), close_loc) if MINE else FUND.iloc[0:0]

def dcf_value(fcf, shares, g, tg, r, years=5):
    if not (fcf > 0 and shares > 0 and r > tg): return np.nan
    cfs = [fcf * (1 + g) ** t for t in range(1, years + 1)]
    pv = sum(cf / (1 + r) ** t for t, cf in enumerate(cfs, 1))
    tv = cfs[-1] * (1 + tg) / (r - tg) / (1 + r) ** years
    return (pv + tv) / shares

D = CONFIG["dcf"]
FUND["dcf_value"] = [dcf_value(row.fcf, row.shares, D["growth_5y"], D["terminal_growth"], D["discount_rate"]) if row.same_ccy else np.nan
                     for row in FUND.itertuples()]
FUND["dcf_upside"] = FUND["dcf_value"] / FUND["price"] - 1

show = ["name", "sector", "currency", "price", "pe", "fwd_pe", "pb", "peg", "div_yield", "fcf_yield", "roe", "margin",
        "debt_equity", "eps_growth", "dcf_value", "dcf_upside"]
display(HTML(f"<h4>Fundamentals & simple DCF (g={D['growth_5y']:.0%}, terminal={D['terminal_growth']:.1%}, r={D['discount_rate']:.0%})</h4>"))
display(FUND[show].style.format({"price": "{:,.2f}", "pe": "{:.1f}", "fwd_pe": "{:.1f}", "pb": "{:.1f}", "peg": "{:.1f}",
        "div_yield": "{:.1%}", "fcf_yield": "{:.1%}", "roe": "{:.0%}", "margin": "{:.0%}", "debt_equity": "{:.2f}",
        "eps_growth": "{:+.0%}", "dcf_value": "{:,.2f}", "dcf_upside": "{:+.0%}"}, na_rep="–")
        .background_gradient(cmap="RdYlGn", subset=["dcf_upside"], vmin=-.8, vmax=.8))

In [ ]:
def dcf_grid(t, growths=None, rates=None):
    row = FUND.loc[t]
    growths = growths if growths is not None else np.arange(.02, .181, .02)
    rates = rates if rates is not None else np.arange(.06, .131, .01)
    z = np.array([[dcf_value(row.fcf, row.shares, g, D["terminal_growth"], r) / row.price - 1 if row.same_ccy else np.nan
                   for g in growths] for r in rates])
    fig = go.Figure(go.Heatmap(z=z, x=[f"{g:.0%}" for g in growths], y=[f"{r:.0%}" for r in rates], colorscale="RdYlGn", zmid=0,
                               zmin=-1, zmax=1, text=[[("" if np.isnan(v) else f"{v:+.0%}") for v in rw] for rw in z],
                               texttemplate="%{text}", colorbar=dict(title="Upside", tickformat=".0%")))
    fig.update_layout(title=f"DCF sensitivity – {label(t)}: upside vs price (rows = discount rate, columns = 5-yr growth)",
                      xaxis_title="5-year FCF growth", yaxis_title="Discount rate", height=430, margin=dict(l=10, r=10, t=70, b=10))
    return fig

valid = FUND.dropna(subset=["dcf_value"]).index
dcf_t = FOCUS if FOCUS in valid else (valid[0] if len(valid) else None)
if dcf_t: dcf_grid(dcf_t).show()
else: print("No stock with a valid DCF (needs positive FCF and matching currency).")

In [ ]:
def zscore(s, w=3):
    s = pd.to_numeric(s, errors="coerce"); sd = s.std()
    return ((s - s.mean()) / sd).clip(-w, w) if sd and not np.isnan(sd) else s * np.nan

def build_scorecard(fund, px, risk, weights):
    c = px[fund.index].ffill()
    k = min(252, len(c) - 1); k1 = min(21, k - 1)
    mom_12_1 = c.iloc[-1 - k1] / c.iloc[-1 - k] - 1
    ret_6m = c.iloc[-1] / c.iloc[-1 - min(126, len(c) - 1)] - 1
    ma200 = c.rolling(min(200, len(c) - 1)).mean().iloc[-1]
    comp = pd.DataFrame(index=fund.index)
    comp["earn_yield"], comp["book_yield"] = fund["earn_yield"], fund["book_yield"]
    comp["fcf_yield"], comp["div_yield"] = fund["fcf_yield"], fund["div_yield"]
    comp["roe"], comp["margin"], comp["low_debt"] = fund["roe"].clip(upper=.6), fund["margin"], -fund["debt_equity"].clip(upper=3)
    comp["mom_12_1"], comp["ret_6m"], comp["vs_ma200"] = mom_12_1, ret_6m, c.iloc[-1] / ma200 - 1
    comp["low_vol"], comp["shallow_dd"] = -risk["Volatility"].reindex(fund.index), risk["Max DD"].reindex(fund.index)
    comp["sharpe"] = risk["Sharpe"].reindex(fund.index)
    groups = {"Value": ["earn_yield", "book_yield", "fcf_yield", "div_yield"], "Quality": ["roe", "margin", "low_debt"],
              "Momentum": ["mom_12_1", "ret_6m", "vs_ma200"], "Risk": ["low_vol", "shallow_dd", "sharpe"]}
    z = comp.apply(zscore)
    fac = pd.DataFrame({g: z[cols].mean(axis=1, skipna=True) for g, cols in groups.items()})
    wts = pd.Series(weights)
    num = fac.mul(wts, axis=1).sum(axis=1, min_count=1)
    den = fac.notna().mul(wts, axis=1).sum(axis=1)
    comp_score = num / den
    out = fac.copy()
    out["Composite z"] = comp_score
    out["Score (0-100)"] = comp_score.rank(pct=True) * 100
    out["Tier"] = pd.cut(out["Score (0-100)"], [-1, 30, 70, 101], labels=["Bottom tier", "Middle", "Top tier"])
    out.insert(0, "Name", [short(t) for t in out.index])
    return out.sort_values("Score (0-100)", ascending=False), comp

SCORE, SCORE_COMP = build_scorecard(FUND, close, RISK, CONFIG["scorecard_weights"])
display(HTML("<h4>Composite scorecard — relative ranking within this stock list</h4>"))
display(SCORE.style.format({c: "{:+.2f}" for c in ["Value", "Quality", "Momentum", "Risk", "Composite z"]} | {"Score (0-100)": "{:.0f}"}, na_rep="–")
        .background_gradient(cmap="RdYlGn", subset=["Value", "Quality", "Momentum", "Risk", "Composite z"], vmin=-1.5, vmax=1.5)
        .background_gradient(cmap="RdYlGn", subset=["Score (0-100)"], vmin=0, vmax=100))

In [ ]:
def scorecard_fig():
    s = SCORE.dropna(subset=["Composite z"]).sort_values("Composite z")
    fig = go.Figure()
    for fct, col in zip(["Value", "Quality", "Momentum", "Risk"], ["#2563eb", "#059669", "#d97706", "#7c3aed"]):
        w = CONFIG["scorecard_weights"][fct]
        fig.add_trace(go.Bar(y=s["Name"], x=s[fct] * w / sum(CONFIG["scorecard_weights"].values()), name=f"{fct} ({w:.0%})",
                             orientation="h", marker_color=col, hovertemplate="%{y}: %{x:+.2f}<extra>" + fct + "</extra>"))
    fig.update_layout(barmode="relative", title="What drives each stock's composite score (weighted factor contributions)",
                      height=60 + 26 * len(s), legend=dict(orientation="h", y=-.12), margin=dict(l=10, r=10, t=60, b=10))
    return fig

def value_quality_fig():
    s = SCORE.dropna(subset=["Value", "Quality"])
    fig = go.Figure(go.Scatter(x=s["Value"], y=s["Quality"], mode="markers+text", text=s["Name"], textposition="top center",
                               textfont=dict(size=10), marker=dict(size=14, color=s["Momentum"], colorscale="RdYlGn", cmid=0,
                               line=dict(width=1, color="white"), colorbar=dict(title="Momentum z"))))
    fig.add_hline(y=0, line=dict(color="#9ca3af", dash="dot")); fig.add_vline(x=0, line=dict(color="#9ca3af", dash="dot"))
    fig.update_layout(title="Value vs Quality (top-right = cheap AND high quality; colour = momentum)", height=520,
                      xaxis_title="Value z-score  →  cheaper", yaxis_title="Quality z-score  →  better business",
                      margin=dict(l=10, r=10, t=60, b=10))
    return fig

scorecard_fig().show(); value_quality_fig().show()

## 🧪 7 · Does the signal actually work? — a simple backtest

Technical rules are easy to *draw* and hard to *prove*. This section tests three classic trend rules on every instrument, against simply **buying and holding**:

| Rule | In the market when… |
|---|---|
| **Price > SMA 200** | the close is above its 200-day average |
| **Golden cross** | SMA 50 is above SMA 200 |
| **MACD > signal** | MACD line is above its signal line (faster, more trades) |
| **2-of-3 vote** | at least two of the three rules agree |

**How the test avoids fooling you**
* **No look-ahead:** a signal seen at today's close is traded at *tomorrow's* return (`signal.shift(1)`).
* **Costs:** every entry/exit pays `bt_cost_bps` (default 0.10 %).
* **Cash is not free money:** when out of the market the strategy earns the risk-free rate.
* **Same window** for the strategy and buy-and-hold (starts when the 200-day average first exists).

**How to read it.** Trend rules rarely beat buy-and-hold on *return* in a bull market; their typical job is **shallower drawdowns** (lower Max DD) and **better Sharpe/Calmar** in crashes. Always check *how many* instruments improved, not just the average.

> ⚠️ Honest caveats: a few years that include a strong bull run is a small sample; rules were not optimised here (good — optimising invites overfitting); taxes, slippage on large orders and whipsaw periods are not modelled. A backtest is evidence, not a promise.

In [ ]:
BT_RULES = ["Buy & hold", "Price > SMA200", "Golden cross", "MACD > signal", "2-of-3 vote"]
COST = CONFIG["bt_cost_bps"] / 1e4

def bt_signals(t):
    df = indicator_frame(t)                              # local prices
    s1 = df["close"] > df["sma200"]; s2 = df["sma50"] > df["sma200"]; s3 = df["macd"] > df["signal"]
    vote = (s1.astype(int) + s2.astype(int) + s3.astype(int)) >= 2
    sig = pd.DataFrame({"Price > SMA200": s1, "Golden cross": s2, "MACD > signal": s3, "2-of-3 vote": vote})
    return sig.where(df["sma200"].notna()), df["sma200"].first_valid_index()

def run_backtest(t):
    sig, start = bt_signals(t)
    r = close[t].pct_change().reindex(sig.index)
    r = r.loc[start:]; sig = sig.loc[start:]
    out = {"Buy & hold": r.fillna(0)}
    pos = {}
    for rule in sig.columns:
        p = sig[rule].shift(1).fillna(False).astype(float)         # trade next day
        cost = p.diff().abs().fillna(0) * COST
        out[rule] = (p * r.fillna(0) + (1 - p) * RF / ANN - cost)
        pos[rule] = p
    pos["Buy & hold"] = pd.Series(1.0, index=r.index)
    return pd.DataFrame(out), pd.DataFrame(pos), start

def perf(ret, pos=None):
    ret = ret.dropna()
    eq = (1 + ret).cumprod()
    yrs = max((ret.index[-1] - ret.index[0]).days / 365.25, 1e-9)
    cg = eq.iloc[-1] ** (1 / yrs) - 1; sd = ret.std() * np.sqrt(ANN)
    res = {"CAGR": cg, "Volatility": sd, "Sharpe": (ret.mean() - RF / ANN) * ANN / sd if sd > 0 else np.nan,
           "Max DD": (eq / eq.cummax() - 1).min()}
    res["Calmar"] = cg / abs(res["Max DD"]) if res["Max DD"] < 0 else np.nan
    if pos is not None:
        res["Time invested"] = pos.mean(); res["Trades / yr"] = pos.diff().abs().sum() / yrs
    return res

BT_RET, BT_POS, BT_ROWS = {}, {}, []
for t in IDX + STK:
    try:
        rr, pp, st = run_backtest(t)
    except Exception:
        continue
    if len(rr) < 250: continue
    BT_RET[t], BT_POS[t] = rr, pp
    for rule in BT_RULES:
        BT_ROWS.append({"ticker": t, "rule": rule, **perf(rr[rule], pp[rule])})
BT = pd.DataFrame(BT_ROWS)

# ---- headline: across ALL instruments, how does each rule compare with buy & hold?
med = BT.groupby("rule")[["CAGR", "Volatility", "Sharpe", "Max DD", "Calmar", "Time invested", "Trades / yr"]].median().loc[BT_RULES]
bh = BT[BT.rule == "Buy & hold"].set_index("ticker")
def share_better(rule, col, higher=True):
    x = BT[BT.rule == rule].set_index("ticker")[col]; b = bh[col].reindex(x.index)
    return float((x > b).mean() if higher else (x > b).mean())      # Max DD is negative: "greater" = shallower
med["% beat B&H on Sharpe"] = [np.nan if r == "Buy & hold" else share_better(r, "Sharpe") for r in med.index]
med["% shallower drawdown"] = [np.nan if r == "Buy & hold" else share_better(r, "Max DD") for r in med.index]
BT_SUMMARY = med
display(HTML(f"<h4>Median across {BT.ticker.nunique()} instruments ({BASE_LABEL}, cost {CONFIG['bt_cost_bps']} bp per trade)</h4>"))
display(BT_SUMMARY.style.format({"CAGR": "{:+.1%}", "Volatility": "{:.1%}", "Sharpe": "{:.2f}", "Max DD": "{:.0%}", "Calmar": "{:.2f}",
        "Time invested": "{:.0%}", "Trades / yr": "{:.1f}", "% beat B&H on Sharpe": "{:.0%}", "% shallower drawdown": "{:.0%}"}, na_rep="–")
        .background_gradient(cmap="RdYlGn", subset=["Sharpe", "Calmar"]))

In [ ]:
def bt_curve_fig(t):
    rr = BT_RET[t]; eq = 100 * (1 + rr).cumprod()
    fig = go.Figure()
    for rule in BT_RULES:
        fig.add_trace(go.Scatter(x=eq.index, y=eq[rule], name=rule, line=dict(width=3 if rule == "Buy & hold" else 1.5,
                      color="#111827" if rule == "Buy & hold" else None)))
    fig.update_layout(title=f"{label(t)} – growth of 100 ({BASE_LABEL}), same start date", height=430, hovermode="x unified",
                      yaxis_type="log", legend=dict(orientation="h", y=-.15), margin=dict(l=10, r=10, t=60, b=10))
    return fig

def bt_dd_fig(t):
    rr = BT_RET[t]; eq = (1 + rr).cumprod(); dd = eq / eq.cummax() - 1
    fig = go.Figure()
    for rule in BT_RULES:
        fig.add_trace(go.Scatter(x=dd.index, y=dd[rule], name=rule, line=dict(width=2.5 if rule == "Buy & hold" else 1.2,
                      color="#111827" if rule == "Buy & hold" else None)))
    fig.update_layout(title=f"{short(t)} – drawdown: how deep did each approach fall?", height=330, yaxis_tickformat=".0%",
                      hovermode="x unified", legend=dict(orientation="h", y=-.2), margin=dict(l=10, r=10, t=60, b=10))
    return fig

BT_T = FOCUS if FOCUS in BT_RET else list(BT_RET)[0]
bt_curve_fig(BT_T).show(); bt_dd_fig(BT_T).show()
bt_one = BT[BT.ticker == BT_T].set_index("rule").loc[BT_RULES].drop(columns="ticker")
display(bt_one.style.format({"CAGR": "{:+.1%}", "Volatility": "{:.1%}", "Sharpe": "{:.2f}", "Max DD": "{:.0%}", "Calmar": "{:.2f}",
                             "Time invested": "{:.0%}", "Trades / yr": "{:.1f}"}))

### Basket test — the whole stock list, equal-weighted
One instrument can be lucky. Here every stock follows the same rule and the results are averaged (equal weight, daily rebalanced), starting when *all* of them have a 200-day average. This is the fairest single picture of whether a rule adds value.

In [ ]:
def basket():
    ts = [t for t in STK if t in BT_RET]
    start = max(BT_RET[t].index[0] for t in ts)
    cols = {rule: pd.concat([BT_RET[t].loc[start:, rule] for t in ts], axis=1).mean(axis=1) for rule in BT_RULES}
    pos = {rule: pd.concat([BT_POS[t].loc[start:, rule] for t in ts], axis=1).mean(axis=1) for rule in BT_RULES}
    return pd.DataFrame(cols), pd.DataFrame(pos), len(ts), start

BK_RET, BK_POS, BK_N, BK_START = basket()
BK_STATS = pd.DataFrame({rule: perf(BK_RET[rule], BK_POS[rule]) for rule in BT_RULES}).T
display(HTML(f"<h4>Equal-weight basket of {BK_N} stocks, from {BK_START.date()} ({BASE_LABEL})</h4>"))
display(BK_STATS.style.format({"CAGR": "{:+.1%}", "Volatility": "{:.1%}", "Sharpe": "{:.2f}", "Max DD": "{:.0%}", "Calmar": "{:.2f}",
                               "Time invested": "{:.0%}", "Trades / yr": "{:.1f}"}, na_rep="–").background_gradient(cmap="RdYlGn", subset=["Sharpe", "Calmar"]))

def basket_fig():
    eq = 100 * (1 + BK_RET).cumprod(); fig = go.Figure()
    for rule in BT_RULES:
        fig.add_trace(go.Scatter(x=eq.index, y=eq[rule], name=rule, line=dict(width=3 if rule == "Buy & hold" else 1.5, color="#111827" if rule == "Buy & hold" else None)))
    fig.update_layout(title=f"Equal-weight basket of {BK_N} stocks – growth of 100 (log scale)", height=430, yaxis_type="log",
                      hovermode="x unified", legend=dict(orientation="h", y=-.15), margin=dict(l=10, r=10, t=60, b=10))
    return fig
basket_fig().show()

## 🖥️ 8 · The dashboard — *The Half Layer markets terminal*

Everything computed above is exported as one JSON payload and rendered by a standalone **terminal-style web app** (`outputs/<universe>_market_dashboard.html`, no Python needed to view it):

* **9 views** on a tracker rail – World · Chart · Signals · Risk · Links · Mix · Value · Test · **Mine** (keys `1`–`9`, `/` jumps to any ticker)
* **CHART = a TradingView-style workstation** (engine: TradingView *Lightweight Charts*): candles / hollow / bars / Heikin Ashi / line / area / baseline, 1D·1W·1M timeframes, **26 indicators** computed live in the browser (SMA/EMA/WMA, Bollinger, Keltner, Donchian, Ichimoku with future cloud, Supertrend, Parabolic SAR, RSI, MACD, Stochastic, Stoch RSI, CCI, Williams %R, ROC, ADX/DMI, Aroon, ATR, historical volatility, Volume, OBV, MFI) each with editable settings and colours, indicator **templates**, symbol **compare** in % mode, **bar replay**, a watchlist with sparklines, log / % / auto scales, snapshot export and fullscreen
* **TECH panel (TradingView-style)**: a needle gauge (Strong sell … Strong buy) from the net vote of 11 oscillators + 15 moving averages, mini gauges for each group, a "More technicals" table with every value and action, pivot points (Classic / Fibonacci / Camarilla / Woodie), and an **analyst rating gauge** with the rating distribution, the **1-year price target** (average, high, low, median, upside) that can be projected on the chart. A *Technical rating* column is added to the Signals board.
* **MINE – your own portfolio** (separate page): add holdings in the page (ticker, shares, average cost, optional purchase date), import / export CSV, or list them in `CONFIG["my_holdings"]` / `data/my_portfolio.csv`. You get portfolio value, unrealised P&L (FX-aware), today's change, beta, volatility, drawdown, allocation by holding / sector / region / currency, time-weighted performance vs the benchmark, risk contribution vs weight, correlations, stress tests, a rebalancing helper, a health check, and TradingView-style technical and analyst ratings per holding. Stocks outside the universe are downloaded the next time the notebook runs and get the full chart workstation.
* **Top-right buttons**: **Refresh** (re-runs this notebook through the local server and reloads), **Sync** (save your holdings, alerts and layout to Google Drive, a folder or a backup file) and **My portfolio**. A small *Developed by Anish Luitel* plate is on every page.
* **Local server** (`serve_dashboard.py`, optional): serves the dashboards on `http://localhost:8765`, adds **intraday timeframes** (1m / 5m / 15m / 1h) and a **Live** mode (refreshes every 20 s and checks your alerts against fresh 1-minute bars), runs the Refresh button, and saves your state to a folder (point it at a Google Drive folder to sync). Start it with `python serve_dashboard.py`.
* **Price alerts**: click the **＋** on the price axis, right-click → *Add alert*, or use the bell tool (Alt+A). Crossing / crossing up / crossing down / greater / less than; draggable lines with bell labels; saved in the browser and **re-checked against new data every time the dashboard is rebuilt**; they also fire bar by bar during **Replay** with a toast and sound.
* **Volume profile**: *Volume Profile (Visible Range)* indicator (up/down volume, POC, value-area high/low, rows and width adjustable, left/right) and a **Fixed Range Volume Profile** drawing tool (Alt+P).
* **Drawing tools**: trend line, ray, horizontal / vertical line, rectangle, Fibonacci retracement, measure, text note – draggable, restylable, with magnet, lock, hide, undo / redo and per-ticker persistence (Alt+T/H/V/F/R/M…)
* **Global filters** by instrument type and region, a live ticker tape, sortable tables, click-through from any row or point to the chart explorer
* **Interactive tools**: risk-dial along the efficient frontier, live DCF sliders that re-price every stock, toggleable backtest rules
* **Plain-English "how to read" notes** on every panel (the *Explain* button hides them)

The look follows The Half Layer brand: deep-navy glass panels over a perspective grid floor, orange for the active element, dashed technical chips, honesty tags, and the signature "half-layer" slicing on headline numbers. The template lives in `dashboard_template/` next to the notebook – edit `style.css` / `app.js` / `pro.js` there to restyle or extend it.

In [ ]:
import base64
import plotly.offline
TEMPLATE_DIR = "dashboard_template"

def _clean(o):
    if isinstance(o, dict):                 return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):        return [_clean(v) for v in o]
    if isinstance(o, (bool, np.bool_)):     return bool(o)
    if isinstance(o, (np.integer,)):        return int(o)
    if isinstance(o, (float, np.floating)): return float(o) if np.isfinite(o) else None
    if isinstance(o, (pd.Timestamp, dt.date, dt.datetime)): return str(o)[:10]
    if o is None or o is pd.NA:             return None
    return o

def _r(v, d=5):
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    return round(v, d) if np.isfinite(v) else None

def _arr(s, d=3):  return [None if pd.isna(v) else round(float(v), d) for v in s]
def _map(row, mapping):  return {new: _r(row[old]) for old, new in mapping.items() if old in row}

RISK_MAP = {"CAGR": "cagr", "Volatility": "vol", "Sharpe": "sharpe", "Sortino": "sortino", "Max DD": "mdd", "Calmar": "calmar",
            "VaR hist": "var_h", "VaR normal": "var_n", "VaR Cornish-Fisher": "var_cf", "CVaR": "cvar", "Skew": "skew",
            "Excess kurt": "kurt", "Worst day": "worst", "Best day": "best"}
CAPM_MAP = {"Beta": "beta", "Alpha (ann.)": "alpha", "Alpha t-stat": "alpha_t", "R²": "r2", "Correlation": "corr",
            "Tracking error": "te", "Info ratio": "ir", "Up capture": "upcap", "Down capture": "dncap"}
BT_MAP = {"CAGR": "cagr", "Volatility": "vol", "Sharpe": "sharpe", "Max DD": "mdd", "Calmar": "calmar", "Time invested": "inv",
          "Trades / yr": "trades", "% beat B&H on Sharpe": "beat", "% shallower drawdown": "shallow"}

def build_payload():
    order = IDX + STK + MINE
    P = {}
    P["meta"] = dict(author="Anish Luitel", universe=CONFIG["universe"], source=DATA_STATUS["source"], is_sample=DATA_STATUS["is_sample"],
                     first=DATA_STATUS["first"], last=DATA_STATUS["last"], n_assets=len(order), currency=BASE_LABEL,
                     base=BASE_CCY, bench=BENCH, rf=RF, conf=CONF, generated=f"{dt.datetime.now():%Y-%m-%d %H:%M}",
                     xfreq="weekly" if XF != "D" else "daily", max_w=max_w, cost_bps=CONFIG["bt_cost_bps"],
                     dcf=dict(CONFIG["dcf"]), weights=dict(CONFIG["scorecard_weights"]), regime=REGIME,
                     pc=[float(x) for x in list(PC_SHARE[:3]) + [np.nan] * (3 - len(PC_SHARE[:3]))], avg_corr=float(avg_corr), focus=FOCUS)
    P["inst"] = [dict(t=t, name=short(t), region=META.loc[t, "region"], type=META.loc[t, "type"], ccy=ticker_ccy(t)[0]) for t in order]
    P["dates"] = [d.strftime("%Y-%m-%d") for d in close.index]
    try:                                              # FX table (units of currency per 1 USD) so costs in any currency can be converted
        _fx = load_fx(["CAD", "EUR", "GBP", "CHF", "JPY", "HKD", "INR", "AUD", "KRW", "CNY", "BRL", "SEK", "NOK", "DKK", "SGD", "MXN", "ZAR", "NZD"], CONFIG, close.index)
        P["fx"] = {c: _arr(_fx[c], 5) for c in _fx.columns}
    except Exception as e:
        P["fx"] = {}
    P["ac"] = {t: _arr(close[t], 4) for t in order}
    # full OHLCV (local prices) for the chart workstation; indicators are computed live in the browser
    opn = MKT["Open"]
    def _ohlc(t):
        c = close_loc[t]
        f = lambda s, d: [None if pd.isna(v) else round(float(v), d) for v in s.reindex(close.index)]
        px = 4 if (c.dropna().median() < 20) else 2
        return {"o": f(opn[t], px), "h": f(high_loc[t], px), "l": f(low_loc[t], px), "c": f(c, px),
                "v": [None if pd.isna(v) else int(v) for v in volume[t].reindex(close.index)]}
    P["ohlc"] = {t: _ohlc(t) for t in order}
    P["tr"] = {t: {k: _r(v) for k, v in TR.loc[t].items()} for t in order}
    P["sig"] = {t: {"p200": _r(r["Price vs SMA200"]), "rsi": _r(r["RSI"], 2), "macd": _r(r["MACD hist %"]), "stoch": _r(r["Stoch %K"], 2),
                    "roc3": _r(r["ROC 3M"]), "mom": _r(r["Mom 12-1"]), "cross": r["Cross state"], "score": int(r["Trend score"])} for t, r in SIG.iterrows()}
    P["risk"] = {t: _map(r, RISK_MAP) | {"uw": int(r["Days under water (max)"])} for t, r in RISK.iterrows()}
    P["capm"] = {t: _map(r, CAPM_MAP) for t, r in CAPM.iterrows()}
    cj = lambda C: {"order": list(C.index), "z": [[_r(v, 3) for v in row] for row in C.values]}
    P["corr"] = {"idx": cj(CORR_IDX), "stk": cj(CORR_STK)}
    ra = xr[[t for t in order if t != BENCH]].rolling(XWIN).corr(xr[BENCH])
    P["roll"] = {"dates": [d.strftime("%Y-%m-%d") for d in ra.index], "s": {t: _arr(ra[t], 3) for t in ra.columns}}
    # efficient frontier
    pts = [dict(vol=_r(v), ret=_r(r), sh=_r((r - RF) / v), w=[_r(x, 4) for x in w]) for (r, v), w in zip(frontier, frontier_w)]
    rs_ = np.random.default_rng(3); keep = rs_.choice(len(cloud), size=min(1500, len(cloud)), replace=False) if len(cloud) else []
    P["frontier"] = dict(tickers=PF_T, names=[short(t) for t in PF_T], mu=[_r(x) for x in mu], sd=[_r(x) for x in np.sqrt(np.diag(S))], pts=pts,
                         cloud=[[_r(cloud[i, 1]), _r(cloud[i, 0]), _r(cloud[i, 2], 3)] for i in keep],
                         port={k2: dict(ret=_r(PORT_STATS.loc[k1, "Exp. return"]), vol=_r(PORT_STATS.loc[k1, "Volatility"]), sh=_r(PORT_STATS.loc[k1, "Sharpe"]),
                                        w=[_r(x, 4) for x in PORT[k1].values]) for k1, k2 in [("Min variance", "minvar"), ("Max Sharpe", "sharpe"), ("Equal weight", "equal")]})
    # valuation
    P["fund"] = {t: dict(name=r["name"], sector=r.get("sector"), ccy=r.get("currency"), price=_r(r["price"], 4), pe=_r(r["pe"]), fwd_pe=_r(r["fwd_pe"]), pb=_r(r["pb"]),
                         div=_r(r["div_yield"]), fcfy=_r(r["fcf_yield"]), roe=_r(r["roe"]), margin=_r(r["margin"]), de=_r(r["debt_equity"]), epsg=_r(r["eps_growth"]),
                         fcf=_r(r["fcf"], 0), shares=_r(r["shares"], 0), same=bool(r["same_ccy"]),
                         an=dict(mean=_r(r.get("rec_mean"), 3), key=(r.get("rec_key") if isinstance(r.get("rec_key"), str) else None), n=_r(r.get("n_analysts"), 0),
                                 tmean=_r(r.get("tgt_mean"), 4), thigh=_r(r.get("tgt_high"), 4), tlow=_r(r.get("tgt_low"), 4), tmed=_r(r.get("tgt_med"), 4),
                                 sb=_r(r.get("rec_strongBuy"), 0), b=_r(r.get("rec_buy"), 0), h=_r(r.get("rec_hold"), 0), s=_r(r.get("rec_sell"), 0), ss=_r(r.get("rec_strongSell"), 0)))
                    for t, r in pd.concat([FUND, FUND_MINE]).iterrows()}
    P["score"] = {t: dict(name=r["Name"], Value=_r(r["Value"], 3), Quality=_r(r["Quality"], 3), Momentum=_r(r["Momentum"], 3), Risk=_r(r["Risk"], 3),
                          comp=_r(r["Composite z"], 3), score=_r(r["Score (0-100)"], 1), tier=str(r["Tier"])) for t, r in SCORE.iterrows()}
    P["holdings"] = HOLDINGS
    # backtest
    to_eq = lambda ret: {rule: _arr((1 + ret[rule]).cumprod().reindex(close.index) * 100, 2) for rule in BT_RULES}
    P["bt"] = dict(rules=BT_RULES, summary={r_: _map(row, BT_MAP) for r_, row in BT_SUMMARY.iterrows()},
                   basket=dict(n=BK_N, start=str(BK_START.date()), stats={r_: _map(row, BT_MAP) for r_, row in BK_STATS.iterrows()}, eq=to_eq(BK_RET)),
                   asset={t: {r_: _map(row, BT_MAP) for r_, row in g.set_index("rule").iterrows()} for t, g in BT.groupby("ticker")},
                   eq={t: to_eq(BT_RET[t]) for t in BT_RET})
    return P

def build_dashboard(path=None):
    if not os.path.isdir(TEMPLATE_DIR):
        raise FileNotFoundError(f"'{TEMPLATE_DIR}/' folder not found next to the notebook – it holds the dashboard template (index.html, style.css, app.js, fonts.css, assets/).")
    path = path or os.path.join(CONFIG["output_dir"], f"{CONFIG['universe']}_market_dashboard.html")
    rd = lambda f: open(os.path.join(TEMPLATE_DIR, f), encoding="utf-8").read()
    logo = "data:image/png;base64," + base64.b64encode(open(os.path.join(TEMPLATE_DIR, "assets", "hl-logo.png"), "rb").read()).decode()
    payload = json.dumps(_clean(build_payload()), separators=(",", ":"), allow_nan=False).replace("</", "<\\/")
    plotly_tag = plotly.offline.get_plotlyjs() if CONFIG["embed_plotlyjs"] else None
    banner = ('<div class="banner">⚠ SAMPLE DATA – numbers are synthetic and meaningless for decisions. Set CONFIG["data_source"] = "yfinance" or "csv".</div>'
              if DATA_STATUS["is_sample"] else "")
    html = rd("index.html")
    html = (html.replace("__TITLE__", f"{CONFIG['universe'].title()} Markets Terminal · The Half Layer").replace("__LOGO__", logo).replace("__BANNER__", banner)
                .replace("/*__FONTS__*/", rd("fonts.css")).replace("/*__CSS__*/", rd("style.css")).replace("/*__APP__*/", rd("app.js"))
                .replace("/*__PRO__*/", rd("pro.js")).replace("/*__MINE__*/", rd("mine.js")).replace("/*__SCREEN__*/", rd("screen.js")).replace("/*__LOCK__*/", rd("lock.js")).replace("/*__BOOT__*/", rd("boot.js")).replace("/*DATAURL*/''", "/*DATAURL*/" + json.dumps(str(CONFIG.get("data_url") or ""))).replace("/*__LIVE__*/", rd("live.js")).replace("/*__SYNC__*/", rd("sync.js")).replace("/*__HELP__*/", rd("help_data.js") + "\n" + rd("help_data2.js") + "\n" + rd("help.js")).replace("/*__LWC__*/", rd("vendor/lightweight-charts.standalone.production.js")))
    html = html.replace("/*__PLOTLY__*/", plotly_tag) if plotly_tag else html.replace("<script>/*__PLOTLY__*/</script>", '<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>')
    html = html.replace("__DATA__", payload)           # last: the payload is the largest string
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    return path

DASH = build_dashboard()
print(f"✅ Dashboard written: {DASH}  ({os.path.getsize(DASH)/1e6:.1f} MB)")

In [ ]:
# Open it right here in the notebook (or just double-click the file in the outputs folder)
from IPython.display import IFrame
IFrame(src=os.path.relpath(DASH), width="100%", height=760)

## 🔁 Switching the data source later

**CSV mode** – create `data/csv/` and drop one file per ticker named like the ticker (`^GSPC` → `_GSPC.csv`, `SHOP.TO` → `SHOP.TO.csv`). Minimum columns: `Date, Close` (optional `Open, High, Low, Volume, Adj Close`). Then set `CONFIG["data_source"] = "csv"`.
For fundamentals add `data/fundamentals.csv` indexed by ticker with the columns `pe, fwd_pe, pb, peg, roe, margin, eps_growth, rev_growth, debt_equity, fcf, shares, mcap, div_rate, currency, fin_currency, sector, industry`.

**API mode** – write a function returning `{"Open": df, "High": df, "Low": df, "Close": df, "Volume": df}` (dates × tickers) and register it:
```python
def provider_alphavantage(tickers, cfg):
    ...                                   # call the API, build the five wide DataFrames
    return {"Open": o, "High": h, "Low": l, "Close": c, "Volume": v}

PROVIDERS["alphavantage"] = provider_alphavantage
CONFIG["data_source"] = "alphavantage"
```

## 📚 Glossary & reading guide

| Term | One-line meaning |
|---|---|
| **Adjusted close** | Price corrected for splits and dividends, so returns are true total returns |
| **Annualise** | Scale daily numbers to a year (mean × 252, volatility × √252) |
| **Drawdown** | Percentage below the previous all-time high |
| **Overbought / oversold** | Price moved up / down unusually fast – a *warning*, not a signal on its own |
| **Beta** | How strongly an asset reacts to the benchmark |
| **Alpha** | Return left over after accounting for market exposure |
| **Shrinkage** | Pulling noisy estimates toward a sensible average so optimisers don't overreact |
| **z-score** | Distance from the group average in standard deviations |
| **Non-synchronous trading** | Different markets close at different times, distorting daily correlations |

## ⚠️ Honest limitations
* FX conversion uses Yahoo daily rates and ignores hedging costs and interest differentials. Switch `base_currency` to `"local"` to see unconverted figures.
* Yahoo fundamentals are **trailing, unaudited summaries**; check filings before relying on them.
* Historical statistics (mean returns, correlations, VaR) **change across regimes** – especially in crises.
* The scorecard is a *relative* ranking inside a 20-stock list, and the DCF is a thought experiment about assumptions.
* This notebook is for **education and research, not personalised financial advice**.